In [ ]:
# CELL 1 — COMMON INFRASTRUCTURE, DATASET ADAPTER, METRICS, TRAINING UTILITIES
import warnings
warnings.filterwarnings("ignore")
warnings.simplefilter("ignore")
import os, re, json, math, copy, random, warnings, subprocess, sys, gc
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import Dict, List, Callable, Optional, Tuple, Any

from scipy.special import softmax as scipy_softmax
from IPython.display import display

try:
    import faiss
    FAISS_AVAILABLE = True
except ImportError:
    FAISS_AVAILABLE = False

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch import Tensor
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torch.amp import autocast, GradScaler

from torchvision import transforms
from sklearn.metrics import roc_auc_score, roc_curve, average_precision_score
from sklearn.cluster import MiniBatchKMeans
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import NearestNeighbors
from scipy import interpolate
from scipy.stats import spearmanr

try:
    from skopt import gp_minimize
    from skopt.space import Real, Integer, Categorical
    SKOPT_AVAILABLE = True
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "scikit-optimize", "-q"], check=False)
try:
    from skopt import gp_minimize
    from skopt.space import Real, Integer, Categorical
    SKOPT_AVAILABLE = True
except ImportError:
    SKOPT_AVAILABLE = False


try:
    import clip
except ImportError as exc:
    raise ImportError(
        "Install OpenAI CLIP first:\n"
        "pip install git+https://github.com/openai/CLIP.git"
    ) from exc

try:
    import open_clip
    OPEN_CLIP_AVAILABLE = True
except ImportError:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "open_clip_torch", "-q"],
        check=False,
    )
    try:
        import open_clip
        OPEN_CLIP_AVAILABLE = True
    except ImportError:
        OPEN_CLIP_AVAILABLE = False
        warnings.warn("open_clip_torch not found — OpenCLIP target will be skipped.")
        warnings.filterwarnings("ignore")

# ----------------------------- USER CONFIG ---------------------------------

@dataclass
class Config:
    dataset_name: str = "Dataset"
    data_root: str = "/./Dataset"
    annotations_root: str = "/./Dataset/Annotations/annotations"
    results_dir: str = "/./Dataset/failure_detection_results22"

    dataset_format: str = "dataset"
    split_file: Optional[str] = None
    image_root: Optional[str] = None
    label_map_file: Optional[str] = None
    val_fraction: float = 0.20

    clip_model: str = "ViT-B/32"
    prompt_template: str = "A photo of a {}."
    siglip_prompt_template: str = "This is a photo of {}."
    logit_scale: float = 100.0

    zero_shot_target_models: Tuple[str, ...] = ("CLIP", "SigLIP", "OpenCLIP")
    siglip_model: str = "google/siglip-base-patch16-224"
    openclip_model: str = "ViT-B-32"
    openclip_pretrained: str = "laion2b_s34b_b79k"
    zeroshot_batch_size: int = 32
    clear_cuda_each_zeroshot_batch: bool = True
    feature_batch_size: int = 256
    num_workers: int = 4
    use_amp: bool = True

    seeds: Tuple[int, ...] = (42, 43, 44, 45, 46)
    max_epochs: int = 100
    patience: int = 15

    # Shared neural-model search protocol.
    # Every neural detector uses this same LR x batch-size grid, validation
    # early stopping, one final retraining, and one test evaluation.
    lr_grid: Tuple[float, ...] = (1e-1, 1e-2, 1e-3, 5e-4, 3e-4)
    batch_size_grid: Tuple[int, ...] = (128, 256, 512, 1024)
    n_iter_freeze_proj: int = 1000
    caption_aux_weight: float = 0.10
    ics_n_subclusters: int = 5
    ics_n_neighbors: int = 20
    ics_freeze_iters: int = 200
    ics_weight_decay: float = 1e-4
    ics_grad_clip: float = 1.0
    ics_fail_batch_ratio: float = 0.50

    # Classical baselines
    auglikelihoodqda_pca_dim: int = 32
    swnc_bo_calls: int = 60
    swnc_max_k: int = 96
    swnc_k1: int = 2
    swnc_k2: int = 5
    swnc_k3: int = 12

    # Ensemble fusion search protocol.
    ensemble_bayes_calls: int = 80
    ensemble_bayes_random_starts: int = 20

CFG = Config()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = bool(CFG.use_amp and torch.cuda.is_available())
ACTIVE_FEATURE_DIM = 512
ACTIVE_TEXT_DIM = 512


def set_active_dims(encoded=None, text_features=None):
    global ACTIVE_FEATURE_DIM, ACTIVE_TEXT_DIM
    if encoded is not None:
        ACTIVE_FEATURE_DIM = int(encoded["train"]["v"].shape[1])
    if text_features is not None:
        ACTIVE_TEXT_DIM = int(text_features.shape[1])

RESULTS_DIR = Path(CFG.results_dir)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# -------------------------- REPRODUCIBILITY --------------------------------

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def clean_device_cache() -> None:
    """Release unreferenced CPU/GPU memory between large model stages."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass

# ------------------------ DATASET-AGNOSTIC API ------------------------------

class DictDataset(Dataset):
    """Wraps any dataset returning (image, label) into a dictionary interface."""
    def __init__(self, base: Dataset):
        self.base = base

    def __len__(self):
        return len(self.base)

    def __getitem__(self, index):
        item = self.base[index]
        if isinstance(item, dict):
            image = item["image"]
            label = item.get("target", item.get("label"))
        else:
            image, label = item[:2]
        return {"image": image, "target": int(label), "index": index}


def _pretty_name(x: str) -> str:
    x = str(x).strip()
    x = Path(x).stem if Path(x).suffix else x
    return " ".join(x.replace("_", " ").replace("-", " ").split()).strip()


def _infer_classname_from_path(path) -> str:
    stem = Path(str(path)).stem
    stem = re.sub(r"_\\d+$", "", stem)
    return _pretty_name(stem)


def _parse_split_record(rec):
    if isinstance(rec, dict):
        relpath = rec.get("impath") or rec.get("img") or rec.get("image") or rec.get("path")
        label = rec.get("label", rec.get("target", rec.get("y")))
        classname = rec.get("classname") or rec.get("class_name") or rec.get("category") or rec.get("name")
        if relpath is None:
            raise ValueError(f"Could not find an image path in split record: {rec}")
        return str(relpath), label, classname
    if isinstance(rec, (list, tuple)):
        if len(rec) >= 3:
            return str(rec[0]), rec[1], rec[2]
        if len(rec) == 2:
            return str(rec[0]), rec[1], None
        if len(rec) == 1:
            return str(rec[0]), None, None
    raise ValueError(f"Unsupported split record: {rec}")


class ZhouSplitDataset(Dataset):
    def __init__(self, records, root, image_root, class_names, label_map=None, transform=None):
        self.transform = transform
        self.class_names = list(class_names)
        self.class_to_idx = {name: i for i, name in enumerate(self.class_names)}
        self.samples = []

        root = Path(root)
        image_root = Path(image_root)

        for rec in records:
            relpath, label, classname = _parse_split_record(rec)
            if label is not None:
                label = int(label)
            if classname is None:
                if label_map is not None and label in label_map:
                    classname = label_map[label]
                else:
                    classname = _infer_classname_from_path(relpath)
            classname = _pretty_name(classname)

            p = Path(relpath)
            candidates = []
            if p.is_absolute():
                candidates.append(p)
            candidates += [
                image_root / p,
                image_root / p.name,
                root / p,
                root / "images" / p,
                root / "images" / p.name,
            ]
            image_path = next((c for c in candidates if c.exists()), None)
            if image_path is None:
                raise FileNotFoundError(f"Could not resolve image path: {relpath}")
            if classname not in self.class_to_idx:
                raise ValueError(f"Unknown class name: {classname}")
            self.samples.append((image_path, self.class_to_idx[classname]))

        self.targets = [y for _, y in self.samples]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        path, target = self.samples[index]
        image = Image.open(path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return {"image": image, "target": target, "index": index, "path": str(path)}


class OfficialDataset(Dataset):
    """Dataset using annotations/{trainval,test}.txt splits."""
    def __init__(self, samples, class_names, transform=None):
        self.samples = list(samples)
        self.class_names = list(class_names)
        self.transform = transform
        self.targets = [int(y) for _, y in self.samples]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        path, target = self.samples[index]
        image = Image.open(path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return {"image": image, "target": int(target), "index": index, "path": str(path)}


def _read_dataset_split_file(split_file: Path):
    records = []
    with open(split_file, "r") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            parts = line.split()
            if len(parts) < 2:
                continue
            stem = parts[0]
            class_id = int(parts[1]) - 1  # official files are 1-indexed
            classname = _pretty_name(re.sub(r"_\d+$", "", stem))
            records.append({"stem": stem, "label": class_id, "classname": classname})
    return records


def _resolve_dataset_image_root(root: Path, cfg: Config) -> Path:
    candidates = []
    if cfg.image_root:
        img = Path(cfg.image_root).expanduser()
        candidates.append(img if img.is_absolute() else root / img)
    candidates.extend([
        root / "images",
        root / "Images" / "images",
        root / "Images",
        root / "dataset" / "images",
        root,
    ])
    for candidate in candidates:
        if candidate.exists() and any(candidate.glob("*.jpg")):
            return candidate
    raise FileNotFoundError(
        "Could not locate Dataset JPG images. Set CFG.image_root to the directory containing *.jpg files."
    )


def _build_dataset_samples(records, image_root: Path):
    samples = []
    for rec in records:
        path = image_root / f"{rec['stem']}.jpg"
        if not path.exists():
            raise FileNotFoundError(f"Missing Dataset image: {path}")
        samples.append((path, int(rec["label"])))
    return samples


def _load_label_map(path):
    if path is None or not Path(path).exists():
        return None
    with open(path, "r") as f:
        obj = json.load(f)
    if not isinstance(obj, dict):
        return None
    out = {}
    for k, v in obj.items():
        try:
            out[int(k)] = _pretty_name(v)
        except Exception:
            pass
    return out or None


def build_dataset_bundle(cfg: Config) -> Dict[str, Any]:
    """
    Automatically supports:
      1. ImageFolder roots with train/val/test directories.
      2. Zhou-style JSON splits with an images directory.
    """
    from torchvision.datasets import ImageFolder

    root = Path(cfg.data_root).expanduser()
    if not root.exists():
        raise FileNotFoundError(
            f"CFG.data_root does not exist: {root}\\n"
            "Set CFG.data_root in Cell 1 to the dataset directory."
        )

    transform = val_transforms
    requested = cfg.dataset_format.lower().strip()

    if requested in {"auto", "dataset", "official_dataset"}:
        annotations_root = Path(cfg.annotations_root).expanduser()
        if not annotations_root.is_absolute():
            annotations_root = root / annotations_root
        trainval_file = annotations_root / "trainval.txt"
        test_file = annotations_root / "test.txt"
        if trainval_file.exists() and test_file.exists():
            trainval_records = _read_dataset_split_file(trainval_file)
            test_records = _read_dataset_split_file(test_file)
            all_records = trainval_records + test_records
            class_names = [None] * (max(r["label"] for r in all_records) + 1)
            for rec in all_records:
                class_names[rec["label"]] = rec["classname"]
            class_names = [name if name is not None else f"class {i}" for i, name in enumerate(class_names)]

            labels = np.array([r["label"] for r in trainval_records])
            train_idx, val_idx = train_test_split(
                np.arange(len(trainval_records)),
                test_size=float(cfg.val_fraction),
                random_state=0,
                stratify=labels,
            )
            image_root = _resolve_dataset_image_root(root, cfg)
            train_records = [trainval_records[i] for i in train_idx]
            val_records = [trainval_records[i] for i in val_idx]
            print("Dataset adapter: Official Dataset annotations")
            print(f"Annotations root: {annotations_root}")
            print(f"Image root: {image_root}")
            print(f"Split sizes: train={len(train_records)}, val={len(val_records)}, test={len(test_records)}")
            return {
                "train": OfficialDataset(_build_dataset_samples(train_records, image_root), class_names, transform),
                "val": OfficialDataset(_build_dataset_samples(val_records, image_root), class_names, transform),
                "test": OfficialDataset(_build_dataset_samples(test_records, image_root), class_names, transform),
                "class_names": class_names,
            }

    folder_layout = all((root / s).is_dir() for s in ("train", "val", "test"))
    if requested in {"auto", "imagefolder"} and folder_layout:
        train = ImageFolder(root / "train", transform=transform)
        val = ImageFolder(root / "val", transform=transform)
        test = ImageFolder(root / "test", transform=transform)
        if train.classes != val.classes or train.classes != test.classes:
            raise ValueError("train, val, and test must share the same class folders.")
        print("Dataset adapter: ImageFolder train/val/test layout")
        return {
            "train": DictDataset(train),
            "val": DictDataset(val),
            "test": DictDataset(test),
            "class_names": [_pretty_name(x) for x in train.classes],
        }

    if cfg.split_file:
        split_path = Path(cfg.split_file).expanduser()
        if not split_path.is_absolute():
            split_path = root / split_path
    else:
        candidates = sorted(root.glob("split_zhou_*.json"))
        split_path = candidates[0] if candidates else None

    if requested in {"auto", "zhou"} and split_path is not None and split_path.exists():
        with open(split_path, "r") as f:
            split_json = json.load(f)

        missing = {"train", "val", "test"}.difference(split_json)
        if missing:
            raise KeyError(f"Split file is missing: {sorted(missing)}")

        image_root = Path(cfg.image_root).expanduser() if cfg.image_root else root / "images"
        if not image_root.is_absolute():
            image_root = root / image_root

        label_map_path = None
        if cfg.label_map_file:
            label_map_path = Path(cfg.label_map_file).expanduser()
            if not label_map_path.is_absolute():
                label_map_path = root / label_map_path
        elif (root / "dataset_label_map.json").exists():
            label_map_path = root / "dataset_label_map.json"

        label_map = _load_label_map(label_map_path)

        class_names = []
        for split in ("train", "val", "test"):
            for rec in split_json[split]:
                relpath, label, classname = _parse_split_record(rec)
                if label is not None:
                    label = int(label)
                if classname is None:
                    if label_map is not None and label in label_map:
                        classname = label_map[label]
                    else:
                        classname = _infer_classname_from_path(relpath)
                classname = _pretty_name(classname)
                if classname not in class_names:
                    class_names.append(classname)

        print(f"Dataset adapter: Zhou split file {split_path.name}")
        print(f"Image root: {image_root}")
        print(f"Classes: {len(class_names)}")

        result = {"class_names": class_names}
        for split in ("train", "val", "test"):
            result[split] = ZhouSplitDataset(
                split_json[split],
                root=root,
                image_root=image_root,
                class_names=class_names,
                label_map=label_map,
                transform=transform,
            )
        return result

    raise RuntimeError(
        "Could not identify the dataset layout. Use either train/val/test class folders "
        "or a Zhou split JSON plus images/."
    )
class ConvertImageToRGB:
    def __call__(self, image):
        return image.convert("RGB") if isinstance(image, Image.Image) else image


def clip_style_transform():
    return transforms.Compose([
        transforms.Resize(224, interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.CenterCrop(224),
        ConvertImageToRGB(),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.48145466, 0.4578275, 0.40821073],
            std=[0.26862954, 0.26130258, 0.27577711],
        ),
    ])


val_transforms = clip_style_transform()

# ----------------------------- METRICS -------------------------------------

def get_auroc(success_scores: np.ndarray, failure_scores: np.ndarray) -> float:
    y = np.r_[np.zeros(len(success_scores)), np.ones(len(failure_scores))]
    s = np.r_[success_scores, failure_scores]
    return float(roc_auc_score(y, s))


def get_fpr95(success_scores: np.ndarray, failure_scores: np.ndarray) -> float:
    y = np.r_[np.zeros(len(success_scores)), np.ones(len(failure_scores))]
    s = np.r_[success_scores, failure_scores]
    fpr, tpr, _ = roc_curve(y, s)
    idx95 = np.where(tpr >= 0.95)[0]
    return float(fpr[idx95[0]]) if len(idx95) > 0 else 1.0


def normalize_uncertainty_scores(scores: np.ndarray) -> np.ndarray:
    scores = np.asarray(scores, dtype=float).reshape(-1)
    lo, hi = np.nanmin(scores), np.nanmax(scores)
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        return np.zeros_like(scores, dtype=float)
    return (scores - lo) / (hi - lo)


def cohen_d_between(success_scores: np.ndarray, failure_scores: np.ndarray) -> float:
    success_scores = np.asarray(success_scores, dtype=float).reshape(-1)
    failure_scores = np.asarray(failure_scores, dtype=float).reshape(-1)
    if len(success_scores) < 2 or len(failure_scores) < 2:
        return float("nan")
    all_scores = np.r_[success_scores, failure_scores]
    all_uncertainty = normalize_uncertainty_scores(all_scores)
    s_u = all_uncertainty[:len(success_scores)]
    f_u = all_uncertainty[len(success_scores):]
    pooled_std = math.sqrt((np.std(f_u) ** 2 + np.std(s_u) ** 2) / 2.0) + 1e-12
    return float((np.mean(f_u) - np.mean(s_u)) / pooled_std)


def youden_threshold_for_uncertainty(scores: np.ndarray, correct_mask: np.ndarray) -> float:
    uncertainty = normalize_uncertainty_scores(scores)
    failure_labels = (~np.asarray(correct_mask, dtype=bool)).astype(int)
    if len(np.unique(failure_labels)) < 2:
        return float("nan")
    fpr, tpr, thresholds = roc_curve(failure_labels, uncertainty)
    j = tpr - fpr
    return float(thresholds[int(np.nanargmax(j))])


def evaluate_failure_scores(
    scores, labels, image_features, text_features, logit_scale=100.0,
    correct_mask=None, target_name="CLIP",
):
    scores = np.asarray(scores, dtype=float).reshape(-1)
    if correct_mask is None:
        with torch.no_grad():
            probs = torch.softmax(logit_scale * image_features @ text_features.T, dim=-1)
            preds = probs.argmax(dim=-1)
            correct = labels.cpu() == preds.cpu()
        correct_np = correct.numpy().astype(bool)
    else:
        correct_np = np.asarray(correct_mask, dtype=bool).reshape(-1)
        if len(correct_np) != len(scores):
            raise ValueError(f"correct_mask length {len(correct_np)} does not match scores length {len(scores)}")
    success = scores[correct_np]
    failure = scores[~correct_np]
    if len(success) == 0 or len(failure) == 0:
        auc = fpr95 = cohen_d = youden_theta = float("nan")
    else:
        auc = get_auroc(success, failure)
        fpr95 = get_fpr95(success, failure)
        cohen_d = cohen_d_between(success, failure)
        youden_theta = youden_threshold_for_uncertainty(scores, correct_np)
    return {
        "target_model": str(target_name),
        "accuracy": float(np.mean(correct_np)),
        "auc": auc,
        "fpr95": fpr95,
        "cohen_d": cohen_d,
        "youden_theta": youden_theta,
        "n_success": int(np.sum(correct_np)),
        "n_failure": int(np.sum(~correct_np)),
        "scores": scores,
        "correct": correct_np,
    }


class FailureBCELoss(nn.Module):
    """Failure-positive BCE for the active zero-shot target model."""
    def forward(self, logits: Tensor, probs: Tensor, labels: Tensor, failure_targets: Optional[Tensor] = None) -> Tensor:
        failure = failure_targets.float() if failure_targets is not None else (labels != probs.argmax(dim=-1)).float()
        n_fail = failure.sum().clamp_min(1.0)
        n_success = (1.0 - failure).sum().clamp_min(1.0)
        pos_weight = (n_success / n_fail).detach()
        return F.binary_cross_entropy_with_logits(
            logits, failure, pos_weight=pos_weight, reduction="mean"
        )


def target_scores_to_probs(scores: Tensor, activation: str = "softmax") -> Tensor:
    if str(activation).lower() == "sigmoid":
        return torch.sigmoid(scores)
    return torch.softmax(scores, dim=-1)


class ViLUWeightedBCELoss(nn.Module):
    """
    Paper-faithful ViLU dynamic log-weighted BCE (Eq. 9-10).

    Positive (failure) samples are up-weighted by:
        w_pos = log(1 + N_success / N_failure)
    computed dynamically per mini-batch. Negative (success) samples carry
    weight 1. This is applied via the per-sample `weight` argument of
    BCEWithLogits (not `pos_weight`), so the log-scaling is exact rather
    than being folded into an internal ratio.
    """
    def forward(self, logits: Tensor, probs: Tensor, labels: Tensor, failure_targets: Optional[Tensor] = None) -> Tensor:
        failure = failure_targets.float() if failure_targets is not None else (labels != probs.argmax(dim=-1)).float()
        n_fail = failure.sum().clamp_min(1.0)
        n_success = (1.0 - failure).sum().clamp_min(1.0)
        # Dynamic log-weighted BCE (paper Eq. 10)
        pos_weight_scalar = torch.log1p(n_success / n_fail).detach()
        weights = torch.where(
            failure > 0.5,
            pos_weight_scalar.expand_as(failure),
            torch.ones_like(failure),
        )
        return F.binary_cross_entropy_with_logits(
            logits, failure, weight=weights, reduction="mean"
        )


def cosine_warmup_lr(base_lr, epoch, total_epochs, warmup=5, min_ratio=0.05):
    if total_epochs <= 0:
        return float(base_lr)
    warmup = max(1, int(warmup))
    if epoch < warmup:
        return float(base_lr) * float(epoch + 1) / float(warmup)
    progress = (epoch - warmup) / max(1, total_epochs - warmup)
    cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
    return float(base_lr) * (float(min_ratio) + (1.0 - float(min_ratio)) * cosine)


def set_lr(optimizer, lr):
    for group in optimizer.param_groups:
        group["lr"] = float(lr) * float(group.get("lr_scale", 1.0))


class UniversalBalancedSampler:
    """Batch sampler that keeps rare CLIP failures visible to ICS training."""
    def __init__(self, labels, fails, batch_size=128, fail_ratio=0.5, n_batches=None, seed=42):
        self.labels = np.asarray(labels)
        self.fails = np.asarray(fails).astype(int)
        self.batch_size = int(batch_size)
        self.n_fail = max(1, int(round(self.batch_size * float(fail_ratio))))
        self.n_success = max(1, self.batch_size - self.n_fail)
        self.n_batches = int(n_batches or max(1, math.ceil(len(self.labels) / self.batch_size)))
        self.seed = int(seed)
        self.fail_map = {}
        self.success_map = {}
        for c in np.unique(self.labels):
            fi = np.where((self.labels == c) & (self.fails == 1))[0]
            si = np.where((self.labels == c) & (self.fails == 0))[0]
            if len(fi) > 0:
                self.fail_map[int(c)] = fi
            if len(si) > 0:
                self.success_map[int(c)] = si
        self.fail_classes = list(self.fail_map.keys())
        self.success_classes = list(self.success_map.keys())
        self._epoch = 0

    def __iter__(self):
        rng = np.random.default_rng(self.seed + self._epoch)
        self._epoch += 1
        for _ in range(self.n_batches):
            batch = []
            if self.fail_classes:
                chosen = rng.choice(self.fail_classes, size=self.n_fail, replace=True)
                batch.extend(int(rng.choice(self.fail_map[int(c)])) for c in chosen)
            if self.success_classes:
                chosen = rng.choice(self.success_classes, size=self.n_success, replace=True)
                batch.extend(int(rng.choice(self.success_map[int(c)])) for c in chosen)
            while len(batch) < self.batch_size:
                batch.append(int(rng.integers(0, len(self.labels))))
            rng.shuffle(batch)
            yield batch[:self.batch_size]

    def __len__(self):
        return self.n_batches


def make_loader(
    v: Tensor, y: Tensor, batch_size: int, seed: int, shuffle=True,
    failure_labels=None, pred_labels=None, target_logits=None,
):
    g = torch.Generator()
    g.manual_seed(seed)
    tensors = [v.float(), y.long()]
    if pred_labels is not None:
        tensors.append(torch.as_tensor(pred_labels).long())
    if failure_labels is not None:
        tensors.append(torch.as_tensor(failure_labels).float())
    if target_logits is not None:
        tensors.append(torch.as_tensor(target_logits).float())
    return DataLoader(
        TensorDataset(*tensors),
        batch_size=batch_size,
        shuffle=shuffle,
        generator=g,
        num_workers=0,
        drop_last=False,
    )


@torch.no_grad()
def score_torch_model(
    model: nn.Module, v: Tensor, t: Tensor, inference_batch_size: int = 1024,
    pred_labels=None, true_labels=None,
) -> np.ndarray:
    model.eval()
    out = []
    bs = max(1, int(inference_batch_size))
    for start in range(0, len(v), bs):
        sl = slice(start, min(start + bs, len(v)))
        z = v[sl].to(DEVICE)
        pb = None if pred_labels is None else torch.as_tensor(pred_labels[sl]).long().to(DEVICE)
        yb = None if true_labels is None else torch.as_tensor(true_labels[sl]).long().to(DEVICE)
        with autocast("cuda", enabled=USE_AMP):
            model_out = model(z, t.to(DEVICE), pred_labels=pb, true_labels=yb)
            logits = model_out[0] if isinstance(model_out, tuple) else model_out
            s = torch.sigmoid(logits.squeeze(-1))
        out.append(s.float().cpu())
    return torch.cat(out).numpy()


def train_one_neural_setting(
    model_builder: Callable[[], nn.Module],
    train_v: Tensor,
    train_y: Tensor,
    val_v: Tensor,
    val_y: Tensor,
    text_features: Tensor,
    seed: int,
    lr: float,
    batch_size: int,
    verbose: bool = True,
    loss_builder: Optional[Callable[[], nn.Module]] = None,
    train_failure_labels=None,
    train_pred_labels=None,
    val_pred_labels=None,
    val_correct_mask=None,
    target_name="CLIP",
    train_target_logits=None,
    val_target_logits=None,
    ):
    """Train one LR/batch-size candidate using validation early stopping only."""
    set_seed(seed)
    model = model_builder().to(DEVICE)
    loss_fn = (loss_builder or FailureBCELoss)()
    optimizer = optim.SGD(model.parameters(), lr=float(lr))
    loader = make_loader(
        train_v, train_y, int(batch_size), seed, shuffle=True,
        failure_labels=train_failure_labels, pred_labels=train_pred_labels,
        target_logits=train_target_logits,
    )
    t = text_features.to(DEVICE)

    best_state = None
    best_auc = -np.inf
    best_fpr = np.inf
    best_epoch = 0
    bad = 0
    history = []

    for epoch in range(CFG.max_epochs):
        model.train()
        losses = []
        for batch in loader:
            cursor = 2
            z_v, labels = batch[0], batch[1]
            if train_pred_labels is not None:
                pred_labels = batch[cursor].to(DEVICE)
                cursor += 1
            else:
                pred_labels = None
            if train_failure_labels is not None:
                failure_targets = batch[cursor].to(DEVICE)
                cursor += 1
            else:
                failure_targets = None
            if train_target_logits is not None:
                batch_target_logits = batch[cursor].to(DEVICE)
            else:
                batch_target_logits = None
            z_v, labels = z_v.to(DEVICE), labels.to(DEVICE)
            with torch.no_grad():
                if batch_target_logits is None:
                    probs = torch.softmax(CFG.logit_scale * z_v @ t.T, dim=-1)
                else:
                    probs = target_scores_to_probs(batch_target_logits, "softmax")
            optimizer.zero_grad(set_to_none=True)
            with autocast("cuda", enabled=USE_AMP):
                model_out = model(z_v, t, pred_labels=pred_labels)
                logits = model_out[0] if isinstance(model_out, tuple) else model_out
                logits = logits.squeeze(-1)
                loss = loss_fn(logits, probs, labels, failure_targets=failure_targets)
                if isinstance(model_out, tuple) and len(model_out) >= 3:
                    cap_p, cap_t = model_out[1], model_out[2]
                    loss = loss + CFG.caption_aux_weight * (1 - (cap_p * cap_t).sum(-1)).mean()
            loss.backward()
            optimizer.step()
            losses.append(float(loss.detach().cpu()))

        val_scores = score_torch_model(
            model, val_v, text_features, inference_batch_size=max(256, batch_size),
            pred_labels=val_pred_labels, true_labels=None,
        )
        val_result = evaluate_failure_scores(
            val_scores, val_y, val_v, text_features, CFG.logit_scale,
            correct_mask=val_correct_mask, target_name=target_name,
        )
        auc = float(val_result["auc"])
        fpr = float(val_result["fpr95"])
        history.append({
            "epoch": epoch + 1,
            "loss": float(np.mean(losses)) if losses else float("nan"),
            "val_auc": auc,
            "val_fpr95": fpr,
        })

        improved = np.isfinite(auc) and (
            auc > best_auc + 1e-12
            or (abs(auc - best_auc) <= 1e-12 and fpr < best_fpr)
        )
        if improved:
            best_auc = auc
            best_fpr = fpr
            best_epoch = epoch + 1
            best_state = copy.deepcopy(model.state_dict())
            bad = 0
        else:
            bad += 1

        if verbose:
            print(
                f"      epoch={epoch+1:03d} "
                f"loss={history[-1]['loss']:.4f} "
                f"val_auc={100*auc:.2f}% "
                f"val_fpr95={100*fpr:.2f}%"
            )
        if bad >= CFG.patience:
            if verbose:
                print(f"      early stopping at epoch {epoch+1}")
            break

    if best_state is None:
        best_state = copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    model.eval()
    return model, {
        "lr": float(lr),
        "batch_size": int(batch_size),
        "best_epoch": int(best_epoch),
        "val_auc": float(best_auc),
        "val_fpr95": float(best_fpr),
        "history": history,
    }


def grid_search_then_train_final(
    model_builder: Callable[[], nn.Module],
    train_v: Tensor,
    train_y: Tensor,
    val_v: Tensor,
    val_y: Tensor,
    text_features: Tensor,
    seed: int,
    fixed_selection: Optional[Dict[str, Any]] = None,
    loss_builder: Optional[Callable[[], nn.Module]] = None,
    train_failure_labels=None,
    train_pred_labels=None,
    val_pred_labels=None,
    val_correct_mask=None,
    target_name="CLIP",
    train_target_logits=None,
    val_target_logits=None,
):
    """
    Shared protocol for ViLU, ATLAS, SOARER, and ICS. Hyperparameters are
    selected on validation once, then can be reused across later seeds.
    """
    search_results = []

    if fixed_selection is None:
        best = None
        for lr in CFG.lr_grid:
            for batch_size in CFG.batch_size_grid:
                print(f"    candidate lr={lr:.0e}, batch_size={batch_size}")
                candidate, info = train_one_neural_setting(
                    model_builder, train_v, train_y, val_v, val_y, text_features,
                    seed=seed, lr=lr, batch_size=batch_size, verbose=False,
                    loss_builder=loss_builder,
                    train_failure_labels=train_failure_labels,
                    train_pred_labels=train_pred_labels,
                    val_pred_labels=val_pred_labels,
                    val_correct_mask=val_correct_mask,
                    target_name=target_name,
                    train_target_logits=train_target_logits,
                    val_target_logits=val_target_logits,
                )
                search_results.append(info)
                if best is None:
                    best = info
                else:
                    better = info["val_auc"] > best["val_auc"] + 1e-12
                    tie = abs(info["val_auc"] - best["val_auc"]) <= 1e-12
                    if better or (tie and info["val_fpr95"] < best["val_fpr95"]):
                        best = info
                del candidate
                clean_device_cache()
                print(
                    f"      validation AUC={100*info['val_auc']:.2f}% | "
                    f"FPR95={100*info['val_fpr95']:.2f}% | "
                    f"best_epoch={info['best_epoch']}"
                )
        print(
            f"    selected lr={best['lr']:.0e}, batch_size={best['batch_size']} "
            f"(val AUC={100*best['val_auc']:.2f}%, "
            f"FPR95={100*best['val_fpr95']:.2f}%)"
        )
    else:
        best = {
            "lr": float(fixed_selection["lr"]),
            "batch_size": int(fixed_selection["batch_size"]),
            "val_auc": float(fixed_selection.get("val_auc", float("nan"))),
            "val_fpr95": float(fixed_selection.get("val_fpr95", float("nan"))),
            "best_epoch": int(fixed_selection.get("best_epoch", 0)),
            "reused_from_grid_seed": fixed_selection.get("grid_seed"),
        }
        print(
            f"    reusing selected lr={best['lr']:.0e}, "
            f"batch_size={best['batch_size']} from the first grid-search seed"
        )

    print("    training one final model with selected hyperparameters...")
    final_model, final_info = train_one_neural_setting(
        model_builder, train_v, train_y, val_v, val_y, text_features,
        seed=seed, lr=best["lr"], batch_size=best["batch_size"], verbose=False,
        loss_builder=loss_builder,
        train_failure_labels=train_failure_labels,
        train_pred_labels=train_pred_labels,
        val_pred_labels=val_pred_labels,
        val_correct_mask=val_correct_mask,
        target_name=target_name,
        train_target_logits=train_target_logits,
        val_target_logits=val_target_logits,
    )
    return final_model, {
        "selected": best,
        "final_validation": final_info,
        "grid": search_results,
    }


def summarize_runs(raw_df: pd.DataFrame) -> pd.DataFrame:
    numeric = ["auc", "fpr95", "cohen_d"]
    rows = []
    for model, grp in raw_df.groupby("model", sort=False):
        row = {"model": model, "n_runs": len(grp)}
        for col in numeric:
            vals = grp[col].astype(float).to_numpy()
            row[f"{col}_mean"] = float(np.nanmean(vals))
            row[f"{col}_variance"] = float(np.nanvar(vals, ddof=1)) if len(vals) > 1 else 0.0
            row[f"{col}_sd"] = float(np.nanstd(vals, ddof=1)) if len(vals) > 1 else 0.0
        rows.append(row)
    return pd.DataFrame(rows)

print(f"Device: {DEVICE} | AMP: {USE_AMP} | Seeds: {CFG.seeds}")
print(f"Neural LR grid: {CFG.lr_grid}")
print(f"Neural batch-size grid: {CFG.batch_size_grid}")

In [ ]:
# CELL 2 — CANONICAL ViLU AND LVU ARCHITECTURES
# ============================================================================
# This cell contains the paper-faithful single canonical architectures for:
# 1. LVU: Visual-only MLP baseline
# 2. ViLU: Triplet-concatenated cross-attention network
# ============================================================================

import math
import torch
import torch.nn as nn

# ============================================================================
# 1. LVU BASELINE (Visual-Only MLP)
# ============================================================================

class LVUNet(nn.Module):
    """
    LVU baseline: A pure visual MLP.
    Constructs representation solely from visual features (z_v).
    """
    def __init__(self, visual_dim=ACTIVE_FEATURE_DIM, textual_dim=ACTIVE_TEXT_DIM, layers=(512, 256, 128, 1)):
        super().__init__()
        self.visual_dim = visual_dim
        self.textual_dim = textual_dim
        
        # Build standard sequential MLP on visual features only
        modules = []
        in_dim = self.visual_dim
        for h_dim in layers[:-1]:
            modules.append(nn.Linear(in_dim, h_dim))
            modules.append(nn.ReLU())
            in_dim = h_dim
        modules.append(nn.Linear(in_dim, layers[-1]))
        
        self.mlp = nn.Sequential(*modules)

    def forward(self, v, t, pred_labels=None):
        # LVU ignores text pool / predictions during failure scoring
        return self.mlp(v)


# ============================================================================
# 2. CANONICAL ViLU ARCHITECTURE (Paper-Faithful Cross-Attention)
# ============================================================================

class ViLURetrievalAttention(nn.Module):
    """
    Canonical ViLU Implementation.
    Fuses visual features with text semantics using the paper-faithful triplet:
    z_ViLU = (z_v, z_t_hat, z_t_alpha)
    
    Where:
    - z_v: Image feature
    - z_t_hat: Emb of the class prediction (or retrieval prediction)
    - z_t_alpha: Attended text pool summary via scaled Q, K, V cross-attention
    """
    def __init__(
        self,
        visual_dim=ACTIVE_FEATURE_DIM,
        textual_dim=ACTIVE_TEXT_DIM,
        proj_dim=None,
        layers=(512, 256, 128, 1),
        logit_scale=100.0,
    ):
        super().__init__()
        self.visual_dim = visual_dim
        self.textual_dim = textual_dim
        self.proj_dim = proj_dim if proj_dim is not None else visual_dim
        self.logit_scale = logit_scale

        # Paper-faithful separate projections for Query, Key, and Value
        self.W_Q = nn.Linear(self.visual_dim, self.proj_dim, bias=False)
        self.W_K = nn.Linear(self.textual_dim, self.proj_dim, bias=False)
        self.W_V = nn.Linear(self.textual_dim, self.proj_dim, bias=False)

        # Input dimension: z_v (visual) + z_t_hat (textual) + z_t_alpha (projected)
        mlp_input_dim = self.visual_dim + self.textual_dim + self.proj_dim

        # Classifier MLP
        modules = []
        in_dim = mlp_input_dim
        for h_dim in layers[:-1]:
            modules.append(nn.Linear(in_dim, h_dim))
            modules.append(nn.ReLU())
            in_dim = h_dim
        modules.append(nn.Linear(in_dim, layers[-1]))
        
        self.mlp = nn.Sequential(*modules)

    def cross_attention(self, z_v, z_t_pool):
        """
        Cross-attention mechanism scaling via 1 / sqrt(d_proj).
        Q = W_Q * z_v
        K = W_K * z_t_pool
        V = W_V * z_t_pool
        """
        Q = self.W_Q(z_v)        # [B, d_proj]
        K = self.W_K(z_t_pool)   # [N, d_proj]
        V = self.W_V(z_t_pool)   # [N, d_proj]

        # Scaled dot-product calculation (1 / sqrt(d))
        scale = 1.0 / math.sqrt(self.proj_dim)
        attn_logits = torch.matmul(Q, K.T) * scale  # [B, N]
        attn_weights = torch.softmax(attn_logits, dim=-1)

        # Final attended context summary (z_t_alpha)
        z_t_alpha = torch.matmul(attn_weights, V)  # [B, d_proj]
        return z_t_alpha

    def forward(self, v, t, pred_labels=None):
        # 1. Obtain predicted text features (z_t_hat)
        if pred_labels is None:
            sims = self.logit_scale * torch.matmul(v, t.T)
            preds = sims.argmax(dim=-1)
        else:
            preds = pred_labels.long().to(v.device)
        z_t_hat = t[preds]  # [B, d_text]

        # 2. Compute attended text summary context (z_t_alpha)
        z_t_alpha = self.cross_attention(v, t)  # [B, d_proj]

        # 3. Concatenate into canonical triplet: (z_v, z_t_hat, z_t_alpha)
        z_ViLU = torch.cat([v, z_t_hat, z_t_alpha], dim=-1)

        # 4. Compute classification failure logit
        return self.mlp(z_ViLU)


# ============================================================================
# 3. MODEL BUILDER FUNCTIONS
# ============================================================================

def build_vilu_model():
    """Builds and returns the canonical ViLU model config."""
    return ViLURetrievalAttention(
        visual_dim=ACTIVE_FEATURE_DIM,
        textual_dim=ACTIVE_TEXT_DIM,
        proj_dim=ACTIVE_FEATURE_DIM,
        layers=(512, 256, 128, 1),
        logit_scale=CFG.logit_scale,
    )


def build_lvu_model():
    """Builds and returns the canonical LVU visual MLP baseline model config."""
    return LVUNet(
        visual_dim=ACTIVE_FEATURE_DIM,
        textual_dim=ACTIVE_TEXT_DIM,
        layers=(512, 256, 128, 1),
    )

In [ ]:
# CELL 3 — CANONICAL ATLAS ARCHITECTURE
# ============================================================================
# This cell contains the canonical implementation of ATLAS.
# Feeds the scorer block with: [projected visual, projected text, CLIP meta].
# ============================================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor

# ============================================================================
# 1. ATLAS UTILITY METADATA EXTRACTORS
# ============================================================================

def _atlas_probs_and_preds(
    v: Tensor,
    t: Tensor,
    logit_scale: float = 100.0,
    pred_labels=None,
    target_logits=None,
    activation: str = "softmax",
    **kwargs,
):
    """
    Unified probability/prediction extraction helper for ATLAS.
    """
    if target_logits is None or str(activation).lower() == "sigmoid":
        probs = torch.softmax((logit_scale * v.float() @ t.float().T), dim=-1)
    else:
        probs = target_scores_to_probs(
            torch.as_tensor(target_logits, device=v.device).float(),
            "softmax",
        ).to(v.device)

    if pred_labels is None:
        preds = probs.argmax(dim=-1)
    else:
        preds = torch.as_tensor(pred_labels, device=v.device, dtype=torch.long)

    return probs, preds


def atlas_clip_meta(
    v: Tensor,
    t: Tensor,
    logit_scale: float = 100.0,
    pred_labels=None,
    target_logits=None,
    activation: str = "softmax",
    **kwargs,
) -> Tensor:
    """
    Extracts 6 CLIP-based heuristic meta-features:
    [1.0 - top1, top1 - top2, entropy, top1, top2, raw_cos]
    """
    probs, preds = _atlas_probs_and_preds(
        v,
        t,
        logit_scale=logit_scale,
        pred_labels=pred_labels,
        target_logits=target_logits,
        activation=activation,
    )

    k = min(2, probs.shape[1])
    top = probs.topk(k=k, dim=-1).values
    top1 = top[:, 0]
    top2 = top[:, 1] if k > 1 else torch.zeros_like(top1)
    entropy = -(probs * (probs + 1e-12).log()).sum(dim=-1)
    
    # Cosine similarity between image and corresponding text predictions
    raw_cos = (v.float() * t[preds].float()).sum(dim=-1)

    return torch.stack([
        1.0 - top1,
        top1 - top2,
        entropy,
        top1,
        top2,
        raw_cos,
    ], dim=1).to(v.dtype)


# ============================================================================
# 2. CANONICAL ATLAS MODEL
# ============================================================================

class ATLASAttention(nn.Module):
    """
    Canonical ATLAS architecture without cross-attention.
    - Learns projection matrices (v_proj, t_proj) with identity warmup.
    - Utilizes Gated Cosine Fusion: pv = sigmoid(<pv, pt>) * pv
    - Feeds final classifier MLP with: [pv, pt, 6-dim CLIP meta features]
    """
    def __init__(self, feat_dim=ACTIVE_FEATURE_DIM, n_iter_freeze_proj=1000, dropout=0.30, clip_meta_dim=6):
        super().__init__()
        self.n_iter_freeze_proj = n_iter_freeze_proj
        self._iter = 0
        self._proj_frozen = True
        self.clip_meta_dim = clip_meta_dim
        self.logit_scale = CFG.logit_scale

        self.v_proj = nn.Linear(feat_dim, feat_dim, bias=False)
        self.t_proj = nn.Linear(feat_dim, feat_dim, bias=False)
        
        # Identity initialization
        nn.init.eye_(self.v_proj.weight)
        nn.init.eye_(self.t_proj.weight)

        # Scorer architecture
        self.scorer = nn.Sequential(
            nn.Linear(feat_dim * 2 + clip_meta_dim, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, 1),
        )
        self.cap_proj = nn.Linear(feat_dim, feat_dim)

    def _set_proj_frozen(self, frozen):
        for p in list(self.v_proj.parameters()) + list(self.t_proj.parameters()):
            p.requires_grad = not frozen
        self._proj_frozen = frozen

    def forward(
        self,
        v,
        t,
        pred_labels=None,
        true_labels=None,
        target_logits=None,
        activation="softmax",
        **kwargs,
    ):
        v = torch.as_tensor(v, dtype=torch.float32)
        t = torch.as_tensor(t, dtype=torch.float32)
        
        if self.training:
            self._iter += 1
            if self._proj_frozen and self._iter >= self.n_iter_freeze_proj:
                self._set_proj_frozen(False)

        probs, preds = _atlas_probs_and_preds(
            v,
            t,
            logit_scale=self.logit_scale,
            pred_labels=pred_labels,
            target_logits=target_logits,
            activation=activation,
        )
        txt = t[preds]

        # Learnable projected modifications
        pv = self.v_proj(v)
        pt = self.t_proj(txt)
        
        # Gated Cosine Fusion logic
        pv = torch.sigmoid((pv * pt).sum(-1, keepdim=True)) * pv

        # Calculate heuristic metadata block
        meta = atlas_clip_meta(
            v,
            t,
            logit_scale=self.logit_scale,
            pred_labels=preds,
            target_logits=target_logits,
            activation=activation,
        )
        
        # Process fused inputs + meta through MLP scorer
        logits = self.scorer(torch.cat([pv, pt, meta], dim=-1))
        
        return logits, F.normalize(self.cap_proj(pv), dim=-1), F.normalize(txt, dim=-1)


# ============================================================================
# 3. BUILDER INTERFACE
# ============================================================================

def build_atlas_model():
    """Builds and returns the canonical non-cross-attention ATLAS model."""
    return ATLASAttention(
        feat_dim=ACTIVE_FEATURE_DIM,
        n_iter_freeze_proj=CFG.n_iter_freeze_proj,
        dropout=0.30,
        clip_meta_dim=6,
    )

In [ ]:
# CELL 4 ? SOARER: WORKING-CODE ARCHITECTURE ADAPTED TO FIXED-SPLIT GRID SEARCH

class SOARERModel(nn.Module):
    """SOARER with learned visual/text semantic projection and frozen CLIP meta features."""
    def __init__(self, feat_dim=512, d_f=256, m_top=5, tau_init=1.0,
                 hidden_dim=256, dropout=0.30, clip_meta_dim=6):
        super().__init__()
        self.feat_dim = feat_dim
        self.m_top = m_top
        self.clip_meta_dim = clip_meta_dim
        self.logit_scale = CFG.logit_scale

        self.W_v = nn.Linear(feat_dim, d_f, bias=False)
        self.W_t = nn.Linear(feat_dim, d_f, bias=False)
        nn.init.xavier_uniform_(self.W_v.weight)
        nn.init.xavier_uniform_(self.W_t.weight)
        self.log_tau = nn.Parameter(torch.tensor(np.log(max(tau_init, 1e-3)), dtype=torch.float32))

        d_sem = (m_top * m_top) + m_top + 2
        input_dim = 2 * feat_dim + d_sem + clip_meta_dim
        self.classifier = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, v, t, pred_labels=None, true_labels=None):
        clip_probs = torch.softmax(self.logit_scale * v @ t.T, dim=-1)
        preds = clip_probs.argmax(dim=-1) if pred_labels is None else pred_labels.long().to(v.device)
        tau = self.log_tau.exp()
        S = self.W_v(v) @ self.W_t(t).T
        p = torch.softmax(S / tau, dim=-1)
        m = min(self.m_top, p.shape[1])
        p_top = p.topk(m, dim=-1, largest=True, sorted=True).values
        if m < self.m_top:
            p_top = F.pad(p_top, (0, self.m_top - m))
        ent = -(p * (p + 1e-12).log()).sum(dim=-1, keepdim=True)
        margin = (p_top[:, :1] - p_top[:, 1:2]) if self.m_top > 1 else torch.zeros_like(ent)
        kron = (p_top.unsqueeze(2) * p_top.unsqueeze(1)).reshape(v.shape[0], -1)
        z_sem = torch.cat([p_top, ent, margin, kron], dim=-1)
        pred_txt = t[preds]
        meta = atlas_clip_meta(v, t, self.logit_scale, pred_labels=preds)
        return self.classifier(torch.cat([v, pred_txt, z_sem, meta], dim=-1))


def build_soarer_model():
    return SOARERModel(feat_dim=ACTIVE_FEATURE_DIM, d_f=256, m_top=5, hidden_dim=256, dropout=0.30)


In [ ]:
# CELL 4 — PREDICTION-CONDITIONED ICS (INTRA-CLASS STRUCTURE) MODEL
# ============================================================================
# This cell contains the prediction-conditioned Intra-Class Structure model.
# Learns projections for visual and textual paths, combined with:
# - inst: High-dimensional localization offset (v - z_t_predicted)
# - cls_feat: 6-dimensional class structural boundary metrics
# ============================================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor

# ============================================================================
# 1. ICS MODEL ARCHITECTURE
# ============================================================================

class ICSModel(nn.Module):
    """
    True prediction-conditioned Intra-Class Structure (ICS) model.
    Learns visual/textual representation projections dynamically while fusing
    instance-level alignment error and class structural statistics.
    """
    def __init__(self, feat_dim=ACTIVE_FEATURE_DIM, freeze_iters=500, dropout=0.30):
        super().__init__()
        self.freeze_iters = freeze_iters
        self._iter = 0
        self._proj_frozen = True
        self.logit_scale = CFG.logit_scale

        # Visual and textual projection blocks
        self.in_proj_v = nn.Sequential(
            nn.Linear(feat_dim, 256),
            nn.LayerNorm(256),
            nn.GELU(),
        )
        self.in_proj_t = nn.Sequential(
            nn.Linear(feat_dim, 256),
            nn.LayerNorm(256),
            nn.GELU(),
        )

        # Fused dimensionality setup:
        # Projected visual (256) + Projected predicted text (256) + inst (feat_dim) + cls_feat (6)
        inst_dim = feat_dim
        cls_dim = 6
        fused_dim = 256 + 256 + inst_dim + cls_dim

        # Deep scorer architecture
        self.scorer = nn.Sequential(
            nn.Linear(fused_dim, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.GELU(),
            nn.Linear(128, 1),
        )
        self.cap_proj = nn.Linear(256, feat_dim)

    def _set_proj_frozen(self, frozen):
        for p in list(self.in_proj_v.parameters()) + list(self.in_proj_t.parameters()):
            p.requires_grad = not frozen
        self._proj_frozen = frozen

    def forward(self, v, t, pred_labels=None, **kwargs):
        v = torch.as_tensor(v, dtype=torch.float32)
        t = torch.as_tensor(t, dtype=torch.float32)

        # Iterative Projection warmup scheduler
        if self.training:
            self._iter += 1
            if self._proj_frozen and self._iter >= self.freeze_iters:
                self._set_proj_frozen(False)

        # 1. Probability mapping & Class prediction resolution
        sims = self.logit_scale * v @ t.T
        probs = torch.softmax(sims, dim=-1)

        if pred_labels is None:
            preds = probs.argmax(dim=-1)
        else:
            preds = torch.as_tensor(pred_labels, device=v.device, dtype=torch.long)
        
        txt = t[preds]  # Predicted text embeddings: [B, feat_dim]

        # 2. Extract localized intra-class offset feature (inst)
        inst = v - txt  # Offset distance in feature space: [B, feat_dim]

        # 3. Extract 6-dim boundary structural features (cls_feat)
        k = min(2, probs.shape[1])
        top = probs.topk(k=k, dim=-1).values
        top1 = top[:, 0]
        top2 = top[:, 1] if k > 1 else torch.zeros_like(top1)
        entropy = -(probs * (probs + 1e-12).log()).sum(dim=-1)
        raw_cos = (v * txt).sum(dim=-1)

        cls_feat = torch.stack([
            1.0 - top1,
            top1 - top2,
            entropy,
            top1,
            top2,
            raw_cos
        ], dim=1)

        # 4. Project features
        pv = self.in_proj_v(v)
        pt = self.in_proj_t(txt)

        # 5. Score representation
        fused = torch.cat([pv, pt, inst, cls_feat], dim=-1)
        logits = self.scorer(fused)

        # Aux Modality alignment reconstruction
        cap_recon = F.normalize(self.cap_proj(pv), dim=-1)
        return logits, cap_recon, F.normalize(txt, dim=-1)


# ============================================================================
# 2. BUILDER FUNCTION
# ============================================================================

def build_ics_model():
    """Builds and returns the canonical prediction-conditioned ICS model."""
    return ICSModel(
        feat_dim=ACTIVE_FEATURE_DIM,
        freeze_iters=CFG.n_iter_freeze_proj,
        dropout=0.30,
    )

In [ ]:
# ============================================================================
# CELL 6 — CANONICAL AUGLIKELIHOODQDA AND SWNC (PIPELINE-FAITHFUL)
#
# Key design matches to pipeline:
#   ✅ AugLikelihoodQDA: PCA → augmented Mahalanobis with shrinkage → 3-feature matrix → fixed LogisticRegression
#   ✅ SWNC: proto-augmented text → dual-view (visual+semantic) → rich features → fixed LogisticRegression
#   ✅ Bank = full pool (train+val combined)
#   ✅ Calibrator trained on same pool (with leave-self-out for SWNC)
#   ✅ Output = failure probability → inverted to confidence for evaluation
#   ✅ All intermediate diagnostics printed
# ============================================================================

import inspect
import numpy as np
import torch
from scipy.special import softmax as scipy_softmax
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

try:
    import faiss
    FAISS_AVAILABLE = True
except ImportError:
    FAISS_AVAILABLE = False


def _np(x):
    """Convert Tensor or array to numpy."""
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy()
    return np.asarray(x)


def build_knn_index(ref):
    """Build a KNN index on L2-normalized vectors. Returns cosine-compatible L2 distances."""
    ref = ref.astype(np.float32)
    refn = ref / (np.linalg.norm(ref, axis=1, keepdims=True) + 1e-8)

    if FAISS_AVAILABLE:
        idx = faiss.IndexFlatIP(refn.shape[1])
        idx.add(refn)

        def qf(q, k):
            q = q.astype(np.float32).reshape(1, -1)
            q = q / (np.linalg.norm(q) + 1e-8)
            sim, ids = idx.search(q, k + 1)
            d = np.sqrt(np.clip(2.0 - 2.0 * sim[0], 0.0, None))
            m = ids[0] >= 0
            return d[m][:k], ids[0][m][:k]
    else:
        def qf(q, k):
            qn = q.astype(np.float32)
            qn = qn / (np.linalg.norm(qn) + 1e-8)
            d = np.linalg.norm(refn - qn, axis=1)
            k1 = min(k + 1, len(d))
            p = np.argpartition(d, k1 - 1)[:k1]
            p = p[np.argsort(d[p])]
            return d[p[:k]], p[:k]

    return qf


def _safe_binary_cv(y, max_cv=5):
    y = np.asarray(y).astype(int)
    bc = np.bincount(y, minlength=2)
    mn = int(bc.min()) if len(bc) >= 2 else 0
    if mn <= 1:
        return 2
    return max(2, min(max_cv, mn))


def fit_binary_calibrator(X, y, Cs=1.0, max_cv=5, max_iter=4000, random_state=42):
    """Fixed train-split calibrator; no internal cross-validation."""
    y = np.asarray(y).astype(int)
    calibrator_c = float(Cs[0] if isinstance(Cs, (list, tuple, np.ndarray)) else Cs)
    print(f"    [CALIBRATOR] samples={len(y)}  fail={int(y.sum())}  "
          f"succ={int((1-y).sum())}  C={calibrator_c:g}  features={X.shape[1]}")
    clf = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            C=calibrator_c,
            class_weight="balanced",
            random_state=random_state,
            max_iter=max_iter,
            solver="lbfgs",
        ),
    )
    clf.fit(X, y)
    train_proba = clf.predict_proba(X)[:, 1]
    try:
        cal_auc = roc_auc_score(y, train_proba)
        print(f"    [CALIBRATOR] train AUC={cal_auc*100:.2f}%")
    except Exception:
        print("    [CALIBRATOR] train AUC=n/a")
    return clf


# ============================================================================
# AUGLIKELIHOODQDA DETECTOR — CANONICAL AugLikelihoodQDA
# ============================================================================

class AugLikelihoodQDA:
    """
    Canonical AugLikelihoodQDA:
      1. PCA reduction
      2. Augmented feature space (PCA + bias)
      3. Per-class covariance with shrinkage toward global
      4. 3-dimensional feature extraction from class log-likelihoods
      5. Fixed logistic calibrator
    """
    def __init__(
        self,
        pca_dim=64,
        whiten=False,
        correct_only=True,
        shrink_alpha=0.05,
        reg=1e-3,
        calibrator_C=1.0,
        seed=42,
    ):
        self.pca_dim = pca_dim
        self.whiten = whiten
        self.correct_only = correct_only
        self.shrink_alpha = shrink_alpha
        self.reg = reg
        self.calibrator_C = calibrator_C
        self.seed = seed

        self.n_classes = None
        self.pca = None
        self.class_S_inv = []
        self.class_logdet = []
        self.calibrator = None

    def fit(self, pool_vis, pool_logits, pool_labels, pool_fail):
        """
        Fit AugLikelihoodQDA bank AND calibrator on the same pool.

        Args:
            pool_vis    : (N, D) visual features
            pool_logits : (N, K) logits (= logit_scale * vis @ txt.T)
            pool_labels : (N,)   ground-truth labels
            pool_fail   : (N,)   1.0 if the active zero-shot target prediction is wrong
        """
        vis = _np(pool_vis).astype(np.float64)
        logits = _np(pool_logits).astype(np.float64)
        labels = _np(pool_labels).astype(np.int64)
        fail = None if pool_fail is None else _np(pool_fail).astype(np.float32)

        self.n_classes = logits.shape[1]

        fail_msg = "raw-density only" if fail is None else f"{int(fail.sum())} failures ({100*fail.mean():.1f}%)"
        print(f"\n  [AugLikelihoodQDA] Fitting on pool: {len(vis)} samples, "
              f"{self.n_classes} classes, {fail_msg}")

        # ── Step 1: PCA ──
        pca_dim_eff = min(self.pca_dim, vis.shape[0] - 1, vis.shape[1])
        print(f"  [AugLikelihoodQDA] PCA: requested={self.pca_dim}, effective={pca_dim_eff}, "
              f"whiten={self.whiten}")
        self.pca = PCA(
            n_components=pca_dim_eff,
            whiten=self.whiten,
            random_state=self.seed,
        )
        X = self.pca.fit_transform(vis)

        # ── Step 2: Augmented space ──
        cp = logits.argmax(axis=1)
        aug = X.shape[1] + 1
        Y = np.concatenate([X, np.ones((len(X), 1), dtype=np.float64)], axis=1)
        Sg = (Y.T @ Y) / len(Y) + self.reg * np.eye(aug)

        # ── Step 3: Per-class covariance with shrinkage ──
        self.class_S_inv = []
        self.class_logdet = []
        n_empty = 0
        n_shrunk = 0

        for c in range(self.n_classes):
            if self.correct_only:
                idx = np.where((labels == c) & (cp == c))[0]
            else:
                idx = np.where(labels == c)[0]

            if len(idx) < 5:
                idx = np.where(labels == c)[0]

            if len(idx) == 0:
                self.class_S_inv.append(np.eye(aug))
                self.class_logdet.append(-1e6)
                n_empty += 1
                continue

            Yc = Y[idx]
            Sc = (Yc.T @ Yc) / max(len(Yc), 1)
            Sc += max(self.reg, 1e-4 * np.trace(Sc) / aug) * np.eye(aug)

            ea = (max(self.shrink_alpha, 0.5 * (1 - len(idx) / (aug * 2)))
                  if len(idx) < aug * 2 else self.shrink_alpha)
            if ea > self.shrink_alpha:
                n_shrunk += 1
            Sc = (1 - ea) * Sc + ea * Sg

            try:
                Si = np.linalg.inv(Sc)
            except np.linalg.LinAlgError:
                Si = np.eye(aug)

            self.class_S_inv.append(Si)
            sign, val = np.linalg.slogdet(Sc)
            self.class_logdet.append(float(val) if sign > 0 else -1e6)

        print(f"  [AugLikelihoodQDA] Covariance: {self.n_classes} classes, "
              f"{n_empty} empty, {n_shrunk} extra-shrunk")

        # ── Step 4: Extract 3-D features ──
        Xfeat = self.raw_matrix(pool_vis, pool_logits)
        print(f"  [AugLikelihoodQDA] Feature matrix: {Xfeat.shape}")

        # ── Step 5: Calibrate only when failure labels are supplied ──
        if fail is not None:
            print(f"  [AugLikelihoodQDA] Calibrating...")
            self.calibrator = fit_binary_calibrator(
                Xfeat, fail, Cs=self.calibrator_C, max_cv=5, max_iter=4000, random_state=self.seed
            )

        return self

    def raw_matrix(self, vis, logits, pred_labels=None):
        """Extract the canonical 3-D likelihood feature vector per sample."""
        vis = _np(vis).astype(np.float64)
        logits = _np(logits).astype(np.float64)

        Xr = self.pca.transform(vis)
        Y = np.concatenate([Xr, np.ones((len(Xr), 1), dtype=np.float64)], axis=1)

        ll = np.zeros((len(Xr), self.n_classes), dtype=np.float64)
        for c in range(self.n_classes):
            ll[:, c] = (
                -0.5 * self.class_logdet[c]
                - 0.5 * np.einsum("ni,ij,nj->n", Y, self.class_S_inv[c], Y)
            )

        preds = logits.argmax(axis=1) if pred_labels is None else np.asarray(pred_labels, dtype=np.int64)
        pred_ll = ll[np.arange(len(preds)), preds]

        tmp = ll.copy()
        tmp[np.arange(len(preds)), preds] = -np.inf
        alt_ll = tmp.max(axis=1)

        probs = scipy_softmax(ll, axis=1)
        feats = np.column_stack([
            (alt_ll - pred_ll).astype(np.float32),
            (1.0 - probs[np.arange(len(preds)), preds]).astype(np.float32),
            (ll.max(axis=1) - pred_ll).astype(np.float32),
        ])
        return feats.astype(np.float32)

    def score_failure_prob(self, vis, logits, pred_labels=None):
        """Return P(failure) for each sample."""
        X = self.raw_matrix(vis, logits, pred_labels=pred_labels)
        proba = self.calibrator.predict_proba(X)[:, 1].astype(np.float32)
        print(f"  [AugLikelihoodQDA] Scored {len(proba)} samples: "
              f"fail_prob min={proba.min():.4f} mean={proba.mean():.4f} max={proba.max():.4f}")
        return proba


# Backward compatibility alias
GMMDetector = AugLikelihoodQDA
AugmentedLikelihoodGMM = AugLikelihoodQDA


# ============================================================================
# SWNC DETECTOR — CANONICAL SWNC
# ============================================================================

class SWNCDetector:
    """
    Canonical SWNC:
      1. Proto-augmented text embeddings
      2. KNN index on bank
      3. Dual-view feature extraction (visual + semantic queries)
      4. Rich per-k features (6 per k per view + 8 similarity + 7 cross-view)
      5. Leave-self-out during calibration
      6. Fixed logistic calibrator
    """
    def __init__(self, k_values=(2, 5, 12), temp=15.0, txt_blend=0.70,
                 calibrator_C=1.0, seed=42):
        self.k_values = sorted(set(k_values))
        self.temp = temp
        self.txt_blend = txt_blend
        self.calibrator_C = calibrator_C
        self.seed = seed

        self.bank_v = None
        self.bank_y = None
        self.proto_np = None
        self.txt_aug = None
        self.q_bank = None
        self.keep_mask = None
        self.calibrator = None
        self.n_classes = None

    def fit(self, pool_vis, pool_labels, text_features, pool_logits, pool_fail):
        """
        Fit SWNC bank AND calibrator on the same pool.
        Leave-self-out is used during calibration feature extraction.

        Args:
            pool_vis       : (N, D) visual features
            pool_labels    : (N,)   ground-truth labels
            text_features  : (K, D) CLIP text embeddings
            pool_logits    : (N, K) logits
            pool_fail      : (N,)   1.0 if the active zero-shot target prediction is wrong
        """
        self.bank_v = _np(pool_vis).astype(np.float32)
        self.bank_y = _np(pool_labels).astype(np.int64)
        txt_np = _np(text_features).astype(np.float32)
        fail = None if pool_fail is None else _np(pool_fail).astype(np.float32)

        self.n_classes = txt_np.shape[0]

        print(f"\n  [SWNC] Fitting on pool: {len(self.bank_v)} samples, "
              f"{self.n_classes} classes, "
              f"{int(fail.sum())} failures ({100*fail.mean():.1f}%)")
        print(f"  [SWNC] k_values={self.k_values}, temp={self.temp}, "
              f"txt_blend={self.txt_blend}")

        # ── Step 1: Proto-augmented text ──
        protos = []
        for c in range(self.n_classes):
            mask = self.bank_y == c
            mu = (self.bank_v[mask].mean(axis=0) if mask.sum() > 0
                  else np.zeros(self.bank_v.shape[1], dtype=np.float32))
            protos.append((mu / (np.linalg.norm(mu) + 1e-12)).astype(np.float32))
        self.proto_np = np.stack(protos)

        alpha = self.txt_blend
        self.txt_aug = alpha * txt_np + (1 - alpha) * self.proto_np
        self.txt_aug = self.txt_aug / (np.linalg.norm(self.txt_aug, axis=1, keepdims=True) + 1e-12)

        # ── Step 2: KNN index ──
        print(f"  [SWNC] Building KNN index on {len(self.bank_v)} samples "
              f"(FAISS={FAISS_AVAILABLE})...")
        self.q_bank = build_knn_index(self.bank_v)

        # ── Step 3: Extract features with leave-self-out ──
        print(f"  [SWNC] Extracting calibration features (leave-self-out)...")
        Xcal = self.extract_feats(
            pool_vis, pool_logits, q_abs=np.arange(len(self.bank_v))
        )

        self.keep_mask = Xcal.std(axis=0) > 1e-6
        n_kept = int(self.keep_mask.sum())
        n_total = len(self.keep_mask)
        print(f"  [SWNC] Features: {n_total} total, {n_kept} kept "
              f"({n_total - n_kept} constant removed)")

        # ── Step 4: Calibrate ──
        print(f"  [SWNC] Calibrating...")
        self.calibrator = fit_binary_calibrator(
            Xcal[:, self.keep_mask],
            fail,
            Cs=self.calibrator_C,
            max_cv=5,
            max_iter=5000,
            random_state=self.seed,
        )

        return self

    def _d2_to_cos(self, d):
        return 1.0 - 0.5 * (np.asarray(d, dtype=np.float32) ** 2)

    def _view_features(self, query_vec, pred, proto_row, q_abs=None):
        """
        Extract per-view features for a single query.
        q_abs: if not None, exclude this index from neighbor results (leave-self-out).
        """
        max_k = max(self.k_values)
        d, ids = self.q_bank(query_vec, min(max_k + 6, len(self.bank_v)))
        ids = np.asarray(ids, dtype=np.int64)
        sims = self._d2_to_cos(d)

        # Leave-self-out
        if q_abs is not None:
            keep = ids != int(q_abs)
            ids = ids[keep]
            sims = sims[keep]

        if len(ids) == 0:
            ids = np.array([0], dtype=np.int64)
            sims = np.array([0.0], dtype=np.float32)

        ids = ids[:max_k]
        sims = sims[:max_k]
        ls = self.bank_y[ids]

        f = []
        for k in self.k_values:
            ke = min(k, len(ids))
            ss = sims[:ke]
            ll = ls[:ke]

            wk = np.exp((ss - ss.max()) / (self.temp + 1e-12))
            wk /= wk.sum() + 1e-12

            counts = np.bincount(ll.astype(np.int64), weights=wk, minlength=self.n_classes)
            pm = counts[pred]
            srt = np.sort(counts)[::-1]
            sec = srt[1] if len(srt) > 1 else 0.0

            f.extend([
                1.0 - pm,
                float(-(counts * np.log(counts + 1e-12)).sum()),
                float(-(pm - (np.delete(counts, pred).max() if len(counts) > 1 else pm))),
                float(1.0 - (ll == pred).mean()),
                float(sec),
                float(np.log1p(pm / (sec + 1e-8) if sec > 0 else pm)),
            ])

        top5 = sims[:5].mean() if len(sims) >= 5 else sims.mean()
        top10 = sims[:10].mean() if len(sims) >= 10 else sims.mean()
        top20 = sims[:20].mean() if len(sims) >= 20 else sims.mean()
        sim_gap = sims[0] - sims[1] if len(sims) > 1 else 0.0
        sim_rel = sims[0] - sims.mean() if len(sims) > 1 else 0.0
        sim_std = sims[:5].std() if len(sims) >= 5 else 0.0

        f.extend([
            float(proto_row[pred]),
            float(proto_row.max() - proto_row[pred]),
            float(top5),
            float(top10),
            float(top20),
            float(sim_gap),
            float(sim_rel),
            float(sim_std),
        ])
        return np.array(f, dtype=np.float32), set(ids[:10]), ls[:10]

    def extract_feats(self, vis, logits_or_preds, q_abs=None):
        """
        Extract dual-view features.
        q_abs: array of absolute indices for leave-self-out (one per query sample).
        """
        qn = _np(vis).astype(np.float32)
        arr = _np(logits_or_preds)
        pn = arr.argmax(axis=1) if arr.ndim == 2 else arr.astype(np.int64)

        # Semantic query: softmax-weighted combination of augmented text
        sq = scipy_softmax((qn @ self.txt_aug.T) * 100.0, axis=1) @ self.txt_aug
        sq /= (np.linalg.norm(sq, axis=1, keepdims=True) + 1e-12)

        npv = len(self.k_values) * 6 + 8   # features per view
        nc = 7                               # cross-view features
        fd = npv * 2 + nc                    # total feature dim
        feats = np.zeros((len(qn), fd), dtype=np.float32)

        pv_all = qn @ self.proto_np.T       # visual proto similarities
        ps_all = sq @ self.proto_np.T        # semantic proto similarities

        for i in range(len(qn)):
            pred = int(pn[i])
            abs_i = None if q_abs is None else int(q_abs[i])

            fv, iv, lv = self._view_features(qn[i], pred, pv_all[i], q_abs=abs_i)
            fs, iset, ls_ = self._view_features(sq[i], pred, ps_all[i], q_abs=abs_i)

            feats[i] = np.array(list(fv) + list(fs) + [
                len(iv & iset) / max(len(iv | iset), 1),
                float((lv == ls_).mean()) if len(lv) and len(ls_) else 0.0,
                float((lv == pred).mean()) if len(lv) else 0.0,
                float((ls_ == pred).mean()) if len(ls_) else 0.0,
                float(np.dot(qn[i], sq[i])),
                abs(float((lv == pred).mean()) - float((ls_ == pred).mean()))
                    if len(lv) and len(ls_) else 0.0,
                min(float((lv == pred).mean()), float((ls_ == pred).mean()))
                    if len(lv) and len(ls_) else 0.0,
            ], dtype=np.float32)

        return feats

    def score_failure_prob(self, vis, logits):
        """Return P(failure) for each sample. No leave-self-out at test time."""
        X = self.extract_feats(vis, logits)
        proba = self.calibrator.predict_proba(X[:, self.keep_mask])[:, 1].astype(np.float32)
        print(f"  [SWNC] Scored {len(proba)} samples: "
              f"fail_prob min={proba.min():.4f} mean={proba.mean():.4f} max={proba.max():.4f}")
        return proba

In [ ]:
# CELL 7 — MCM, CLIP FEATURE EXTRACTION, FIXED-SPLIT MULTI-SEED EXPERIMENT, VARIANCE/SD
# REVISED: all variant sweeps removed; each model runs as a single canonical configuration.

@torch.no_grad()
def encode_text_features(clip_model, class_names):
    prompts = [CFG.prompt_template.format(name) for name in class_names]
    tokens = clip.tokenize(prompts).to(DEVICE)
    with autocast("cuda", enabled=USE_AMP):
        t = clip_model.encode_text(tokens)
        t = t / t.norm(dim=-1, keepdim=True)
    return t.float().cpu()


@torch.no_grad()
def encode_split(clip_model, dataset, split_name):
    loader = DataLoader(
        dataset,
        batch_size=CFG.feature_batch_size,
        shuffle=False,
        num_workers=CFG.num_workers,
        pin_memory=torch.cuda.is_available(),
    )
    features, labels = [], []
    clip_model.eval()
    for batch in loader:
        images = batch["image"].to(DEVICE)
        y = torch.as_tensor(batch["target"]).long()
        with autocast("cuda", enabled=USE_AMP):
            v = clip_model.encode_image(images)
            v = v / v.norm(dim=-1, keepdim=True)
        features.append(v.float().cpu())
        labels.append(y.cpu())
    return torch.cat(features), torch.cat(labels)


def load_or_encode_features(bundle):
    dataset_tag = f"{CFG.dataset_name}_{CFG.dataset_format}_{CFG.clip_model.replace('/', '-')}"
    cache = RESULTS_DIR / f"{dataset_tag}_features.pt"
    split_lengths = {split: len(bundle[split]) for split in ("train", "val", "test")}
    if cache.exists():
        obj = torch.load(cache, map_location="cpu", weights_only=False)
        cached_lengths = obj.get("split_lengths")
        if obj.get("class_names") == list(bundle["class_names"]) and cached_lengths == split_lengths:
            print(f"Loaded feature cache: {cache}")
            return obj["encoded"], obj["text_features"]

    clip_model, _ = clip.load(CFG.clip_model, device=DEVICE)
    clip_model.eval()
    text_features = encode_text_features(clip_model, bundle["class_names"])
    encoded = {}
    for split in ("train", "val", "test"):
        encoded[split] = {}
        encoded[split]["v"], encoded[split]["y"] = encode_split(
            clip_model, bundle[split], split
        )
    del clip_model
    clean_device_cache()
    torch.save(
        {
            "encoded": encoded,
            "text_features": text_features,
            "class_names": list(bundle["class_names"]),
            "split_lengths": split_lengths,
            "config": asdict(CFG),
        },
        cache,
    )
    print(f"Saved feature cache: {cache}")
    return encoded, text_features


def _dataset_image_paths(dataset):
    if hasattr(dataset, "samples"):
        return [str(p) for p, _ in dataset.samples]
    base = getattr(dataset, "base", None)
    if base is not None and hasattr(base, "samples"):
        return [str(p) for p, _ in base.samples]
    paths = []
    for i in range(len(dataset)):
        item = dataset[i]
        if "path" not in item:
            raise ValueError("Target zero-shot evaluation needs image paths; dataset item has no 'path'.")
        paths.append(str(item["path"]))
    return paths


def _open_rgb_images(paths):
    images = []
    for p in paths:
        with Image.open(p) as img:
            images.append(img.convert("RGB"))
    return images


def _target_accuracy_record(logits, labels, target_name, text_features=None, image_features=None, activation="softmax"):
    logits = np.asarray(logits, dtype=np.float32)
    labels_np = _np(labels).astype(np.int64)
    preds = logits.argmax(axis=1).astype(np.int64)
    correct = preds == labels_np
    rec = {
        "logits": logits,
        "labels": labels_np,
        "preds": preds,
        "correct": correct,
        "accuracy": float(correct.mean()),
        "target_model": str(target_name),
        "activation": str(activation),
    }
    if text_features is not None:
        rec["text_features"] = _np(text_features).astype(np.float32)
    if image_features is not None:
        rec["image_features"] = image_features.float().cpu() if isinstance(image_features, torch.Tensor) else torch.as_tensor(image_features).float()
    return rec


def _clip_target_predictions(encoded, text_features):
    txt = text_features.float().cpu()
    return {split: _target_accuracy_record(_clip_split_logits(encoded[split]["v"], txt), encoded[split]["y"], "CLIP", text_features=txt, image_features=encoded[split]["v"], activation="softmax") for split in ("train", "val", "test")}


def _hf_pooled_tensor(output):
    if isinstance(output, torch.Tensor):
        return output
    for attr in ("pooler_output", "image_embeds", "text_embeds", "last_hidden_state"):
        value = getattr(output, attr, None)
        if isinstance(value, torch.Tensor):
            return value[:, 0] if value.ndim == 3 else value
    if isinstance(output, (tuple, list)):
        for value in output:
            if isinstance(value, torch.Tensor) and value.ndim == 2:
                return value
    raise TypeError(f"Could not extract a pooled tensor from {type(output).__name__}")


def _normalize_embedding(output):
    emb = _hf_pooled_tensor(output)
    return emb / emb.norm(dim=-1, keepdim=True).clamp_min(1e-12)


@torch.no_grad()
def _siglip_split_logits(model, processor, dataset, class_names, split_name):
    template = getattr(CFG, "siglip_prompt_template", "This is a photo of {}.")
    prompts = [template.format(name) for name in class_names]
    paths = _dataset_image_paths(dataset)
    rows, feats = [], []
    text_features = None
    bs = max(1, int(CFG.zeroshot_batch_size))

    for start in range(0, len(paths), bs):
        images = _open_rgb_images(paths[start:start+bs])
        inputs = processor(
            text=prompts, images=images,
            padding="max_length", truncation=True, return_tensors="pt",
        ).to(DEVICE)
        with autocast("cuda", enabled=USE_AMP):
            out = model(**inputs)
            scores = torch.sigmoid(out.logits_per_image)
            image_features = F.normalize(out.image_embeds.float(), dim=-1)
            if text_features is None:
                text_features = F.normalize(out.text_embeds.float(), dim=-1).detach().cpu()
        if scores.shape != (len(images), len(prompts)):
            raise ValueError(
                f"SigLIP scores for {split_name} have shape {tuple(scores.shape)}; "
                f"expected {(len(images), len(prompts))}."
            )
        rows.append(scores.float().cpu())
        feats.append(image_features.float().cpu())
        del images, inputs, out, image_features, scores
        if CFG.clear_cuda_each_zeroshot_batch:
            clean_device_cache()
        print(f"  SigLIP {split_name}: {min(start+bs, len(paths))}/{len(paths)}", end="\r")

    print()
    clean_device_cache()
    return (
        torch.cat(rows, dim=0).numpy().astype(np.float32),
        text_features.numpy().astype(np.float32),
        torch.cat(feats, dim=0),
    )


@torch.no_grad()
def _openclip_split_logits(model, tokenizer, preprocess, dataset, class_names, split_name):
    prompts = [CFG.prompt_template.format(name) for name in class_names]
    tokens = tokenizer(prompts).to(DEVICE)
    with autocast("cuda", enabled=USE_AMP):
        text_features = model.encode_text(tokens)
        text_features = F.normalize(text_features.float(), dim=-1)
    text_features = text_features.cpu()
    print(f"  OpenCLIP {split_name}: text features encoded {text_features.shape}")

    paths = _dataset_image_paths(dataset)
    rows, feats = [], []
    bs = max(1, int(CFG.zeroshot_batch_size))

    for start in range(0, len(paths), bs):
        batch_paths = paths[start:start + bs]
        images = _open_rgb_images(batch_paths)
        image_tensors = torch.stack([preprocess(img) for img in images]).to(DEVICE)
        with autocast("cuda", enabled=USE_AMP):
            image_features = model.encode_image(image_tensors)
            image_features = F.normalize(image_features.float(), dim=-1)
        logits = 100.0 * image_features.cpu() @ text_features.T
        rows.append(logits.float())
        feats.append(image_features.float().cpu())
        del images, image_tensors, image_features, logits
        if CFG.clear_cuda_each_zeroshot_batch:
            clean_device_cache()
        print(f"  OpenCLIP {split_name}: {min(start + bs, len(paths))}/{len(paths)}", end="\r")

    print()
    clean_device_cache()
    return torch.cat(rows, dim=0).numpy().astype(np.float32), text_features.numpy().astype(np.float32), torch.cat(feats, dim=0)


def canonical_target_model_names(names):
    aliases = {
        "clip": "CLIP", "siglip": "SigLIP", "blip": "BLIP",
        "openclip": "OpenCLIP", "open_clip": "OpenCLIP",
    }
    out = []
    for name in names:
        key = str(name).strip().lower().replace("-", "").replace("_", "")
        matched = aliases.get(key) or aliases.get(str(name).strip().lower())
        if not key:
            continue
        if matched is None:
            raise ValueError(
                f"Unknown zero-shot target model: {name}. "
                f"Expected one of {sorted(set(aliases.values()))}"
            )
        if matched not in out:
            out.append(matched)
    return tuple(out)


def load_or_compute_target_predictions(bundle, encoded, text_features):
    requested = canonical_target_model_names(CFG.zero_shot_target_models)
    dataset_tag = f"{CFG.dataset_name}_{CFG.dataset_format}_{CFG.clip_model.replace('/', '-')}"
    cache = RESULTS_DIR / f"{dataset_tag}_zeroshot_targets.pt"
    split_lengths = {split: len(bundle[split]) for split in ("train", "val", "test")}
    cache_key = {
        "requested": requested,
        "class_names": list(bundle["class_names"]),
        "split_lengths": split_lengths,
        "siglip_model": CFG.siglip_model,
        "openclip_model": getattr(CFG, "openclip_model", ""),
        "openclip_pretrained": getattr(CFG, "openclip_pretrained", ""),
        "prompt_template": CFG.prompt_template,
        "siglip_prompt_template": getattr(CFG, "siglip_prompt_template", "This is a photo of {}."),
        "target_logit_pipeline_version": 10,
    }
    if cache.exists():
        obj = torch.load(cache, map_location="cpu", weights_only=False)
        if obj.get("cache_key") == cache_key:
            print(f"Loaded zero-shot target cache: {cache}")
            return obj["targets"]
    targets = {}
    if "CLIP" in requested:
        targets["CLIP"] = _clip_target_predictions(encoded, text_features)
    if any(m == "SigLIP" for m in requested):
        try:
            from transformers import AutoModel, AutoProcessor
        except ImportError as exc:
            raise ImportError("SigLIP zero-shot target requires transformers.") from exc

        hf_model_kwargs = (
            {"torch_dtype": torch.float16, "low_cpu_mem_usage": True}
            if torch.cuda.is_available() and USE_AMP
            else {}
        )

        if "SigLIP" in requested:
            clean_device_cache()
            print(f"Loading SigLIP target model: {CFG.siglip_model}")
            processor = AutoProcessor.from_pretrained(CFG.siglip_model, use_fast=True)
            model = AutoModel.from_pretrained(CFG.siglip_model, **hf_model_kwargs).to(DEVICE).eval()
            targets["SigLIP"] = {}
            for split in ("train", "val", "test"):
                logits, siglip_text_features, siglip_image_features = _siglip_split_logits(
                    model, processor, bundle[split], bundle["class_names"], split
                )
                targets["SigLIP"][split] = _target_accuracy_record(
                    logits, encoded[split]["y"], "SigLIP",
                    text_features=siglip_text_features,
                    image_features=siglip_image_features, activation="sigmoid"
                )
            del model, processor
            clean_device_cache()

    if "OpenCLIP" in requested:
        if not OPEN_CLIP_AVAILABLE:
            raise ImportError("OpenCLIP target requires open_clip_torch.")
        clean_device_cache()
        print(f"Loading OpenCLIP target model: {CFG.openclip_model} ({CFG.openclip_pretrained})")
        oc_model, _, oc_preprocess = open_clip.create_model_and_transforms(
            CFG.openclip_model, pretrained=CFG.openclip_pretrained, device=DEVICE,
        )
        oc_model = oc_model.eval()
        if USE_AMP and torch.cuda.is_available():
            oc_model = oc_model.half()
        oc_tokenizer = open_clip.get_tokenizer(CFG.openclip_model)
        targets["OpenCLIP"] = {}
        for split in ("train", "val", "test"):
            logits, openclip_text_features, openclip_image_features = _openclip_split_logits(
                oc_model, oc_tokenizer, oc_preprocess, bundle[split], bundle["class_names"], split
            )
            targets["OpenCLIP"][split] = _target_accuracy_record(
                logits, encoded[split]["y"], "OpenCLIP",
                text_features=openclip_text_features,
                image_features=openclip_image_features, activation="softmax"
            )
        del oc_model, oc_tokenizer, oc_preprocess
        clean_device_cache()
    torch.save({"cache_key": cache_key, "targets": targets}, cache)
    print(f"Saved zero-shot target cache: {cache}")
    return targets


def make_target_context(target_predictions, target_name):
    target = target_predictions[target_name]
    return {
        "name": str(target_name),
        "text_features": torch.as_tensor(target["train"].get("text_features"), dtype=torch.float32),
        "activation": str(target["train"].get("activation", "softmax")),
        "encoded": {
            "train": {"v": target["train"].get("image_features"), "y": torch.as_tensor(target["train"].get("labels"), dtype=torch.long)},
            "val": {"v": target["val"].get("image_features"), "y": torch.as_tensor(target["val"].get("labels"), dtype=torch.long)},
            "test": {"v": target["test"].get("image_features"), "y": torch.as_tensor(target["test"].get("labels"), dtype=torch.long)},
        },
        "train_logits": np.asarray(target["train"]["logits"], dtype=np.float32),
        "val_logits": np.asarray(target["val"]["logits"], dtype=np.float32),
        "test_logits": np.asarray(target["test"]["logits"], dtype=np.float32),
        "train_preds": np.asarray(target["train"]["preds"], dtype=np.int64),
        "val_preds": np.asarray(target["val"]["preds"], dtype=np.int64),
        "test_preds": np.asarray(target["test"]["preds"], dtype=np.int64),
        "train_correct": np.asarray(target["train"]["correct"], dtype=bool),
        "val_correct": np.asarray(target["val"]["correct"], dtype=bool),
        "test_correct": np.asarray(target["test"]["correct"], dtype=bool),
        "train_failure": (~np.asarray(target["train"]["correct"], dtype=bool)).astype(np.float32),
        "val_failure": (~np.asarray(target["val"]["correct"], dtype=bool)).astype(np.float32),
        "test_failure": (~np.asarray(target["test"]["correct"], dtype=bool)).astype(np.float32),
    }


def clip_predictions(v, t, target_logits=None, activation="softmax"):
    if target_logits is None or activation == "sigmoid":
        p = torch.softmax(CFG.logit_scale * v @ t.T, dim=-1)
    else:
        p = target_scores_to_probs(torch.as_tensor(target_logits).float(), "softmax")
    return p, p.argmax(dim=-1)


def run_mcm(split_v, split_y, text_features, target_correct=None, target_name="CLIP", target_preds=None, target_logits=None, activation="softmax"):
    probs, _ = clip_predictions(split_v, text_features, target_logits=target_logits, activation=activation)
    if target_preds is None:
        scores = (1.0 - probs.max(dim=-1).values).numpy()
    else:
        idx = torch.as_tensor(target_preds, dtype=torch.long)
        scores = (1.0 - probs[torch.arange(len(idx)), idx]).numpy()
    return evaluate_failure_scores(scores, split_y, split_v, text_features, CFG.logit_scale, correct_mask=target_correct, target_name=target_name)


def run_neural_model(name, builder, encoded, text_features, seed, fixed_selection=None, loss_builder=None, target_context=None):
    mode = "validation-only LR/batch-size grid search" if fixed_selection is None else "fixed LR/batch-size from first grid-search seed"
    train_target_logits = None if target_context is None else target_context["train_logits"]
    val_target_logits = None if target_context is None else target_context["val_logits"]

    print(f"\n  [{name}] {mode}")
    model, selection_info = grid_search_then_train_final(
        builder,
        encoded["train"]["v"], encoded["train"]["y"],
        encoded["val"]["v"], encoded["val"]["y"],
        text_features,
        seed=seed,
        fixed_selection=fixed_selection,
        loss_builder=loss_builder,
        train_failure_labels=None if target_context is None else target_context["train_failure"],
        train_pred_labels=None if target_context is None else target_context["train_preds"],
        val_pred_labels=None if target_context is None else target_context["val_preds"],
        val_correct_mask=None if target_context is None else target_context["val_correct"],
        target_name="CLIP" if target_context is None else target_context["name"],
        train_target_logits=train_target_logits,
        val_target_logits=val_target_logits,
    )

    selected_bs = int(selection_info["selected"]["batch_size"])
    train_scores = score_torch_model(
        model, encoded["train"]["v"], text_features,
        inference_batch_size=max(256, selected_bs),
        pred_labels=None if target_context is None else target_context["train_preds"],
        true_labels=None,
    )
    train_result = evaluate_failure_scores(
        train_scores, encoded["train"]["y"], encoded["train"]["v"],
        text_features, CFG.logit_scale,
        correct_mask=None if target_context is None else target_context["train_correct"],
        target_name="CLIP" if target_context is None else target_context["name"],
    )
    val_scores = score_torch_model(
        model, encoded["val"]["v"], text_features,
        inference_batch_size=max(256, selected_bs),
        pred_labels=None if target_context is None else target_context["val_preds"],
        true_labels=None,
    )
    val_result = evaluate_failure_scores(
        val_scores, encoded["val"]["y"], encoded["val"]["v"],
        text_features, CFG.logit_scale,
        correct_mask=None if target_context is None else target_context["val_correct"],
        target_name="CLIP" if target_context is None else target_context["name"],
    )

    scores = score_torch_model(
        model, encoded["test"]["v"], text_features,
        inference_batch_size=max(256, selected_bs),
        pred_labels=None if target_context is None else target_context["test_preds"],
        true_labels=None,
    )
    result = evaluate_failure_scores(
        scores, encoded["test"]["y"], encoded["test"]["v"],
        text_features, CFG.logit_scale,
        correct_mask=None if target_context is None else target_context["test_correct"],
        target_name="CLIP" if target_context is None else target_context["name"],
    )
    target_tag = "CLIP" if target_context is None else target_context["name"]
    ckpt = RESULTS_DIR / f"{CFG.dataset_name}_{target_tag}_{name}_seed{seed}.pt"
    torch.save(
        {
            "state_dict": model.state_dict(),
            "seed": seed,
            "selection": selection_info,
            "train_result": train_result,
            "validation_result": val_result,
            "test_result": result,
            "config": asdict(CFG),
        },
        ckpt,
    )
    del model
    clean_device_cache()
    return result, val_result, train_result, selection_info


def _clip_pred_np(v, text_features):
    logits = _clip_split_logits(v, text_features)
    return logits.argmax(axis=1).astype(np.int64), logits


def build_ics_manifold(train_v, train_y, text_features, seed, train_logits=None,
                       train_preds_override=None, correct_only_manifold=False,
                       min_correct_per_class=5, variant_name="ICS"):
    print(f"  Building ICS manifold ({variant_name}): "
          f"TRUE class statistics, correct_only={correct_only_manifold}")
    txt = text_features.float().cpu()
    n_classes = txt.shape[0]
    feat_dim = train_v.shape[1]
    n_sub = int(CFG.ics_n_subclusters)

    cls_mu = torch.zeros(n_classes, feat_dim)
    cls_sigma = torch.zeros(n_classes)
    cls_deff = torch.zeros(n_classes)
    cls_sub_mu = torch.zeros(n_classes, n_sub, feat_dim)
    cls_sub_sig = torch.zeros(n_classes, n_sub)
    cls_sub_pi = torch.zeros(n_classes, n_sub)
    cls_var_diag = torch.zeros(n_classes, feat_dim)
    cls_knn = {}
    cls_fvec = {}
    cls_cos_stats = {}
    fallback_classes = []

    labels_np = _np(train_y).astype(np.int64)
    preds_np = (None if train_logits is None else np.argmax(_np(train_logits), axis=1).astype(np.int64))
    if train_preds_override is not None:
        preds_np = np.asarray(train_preds_override, dtype=np.int64)
    for c in range(n_classes):
        all_idx = np.where(labels_np == c)[0]
        if correct_only_manifold and preds_np is not None:
            idx = np.where((labels_np == c) & (preds_np == c))[0]
            if len(idx) < int(min_correct_per_class):
                idx = all_idx
                fallback_classes.append(c)
        else:
            idx = all_idx
        Xc = train_v[idx].float().cpu()
        nc = Xc.shape[0]
        if nc == 0:
            continue
        mu = Xc.mean(0)
        cls_mu[c] = mu
        diff = Xc - mu
        cls_var_diag[c] = diff.pow(2).mean(0)
        cls_sigma[c] = diff.norm(dim=-1).pow(2).mean().sqrt()
        if nc > 1:
            try:
                _, S, _ = torch.linalg.svd(diff, full_matrices=False)
                lam = S.pow(2) / nc
                cls_deff[c] = lam.sum().pow(2) / (lam.pow(2).sum() + 1e-8)
            except Exception:
                cls_deff[c] = 0.0

        ns_eff = min(n_sub, max(1, nc))
        if nc >= ns_eff:
            km = MiniBatchKMeans(ns_eff, random_state=seed, n_init=5, batch_size=min(1024, nc))
            km.fit(Xc.numpy())
            for j in range(ns_eff):
                sj = Xc[km.labels_ == j]
                if sj.shape[0] == 0:
                    continue
                muj = sj.mean(0)
                cls_sub_mu[c, j] = muj
                cls_sub_sig[c, j] = (sj - muj).norm(dim=-1).mean()
                cls_sub_pi[c, j] = float(sj.shape[0] / nc)

        cls_knn[c] = build_knn_index(Xc.numpy())
        cls_fvec[c] = Xc.clone()

        bank_cos_c = F.cosine_similarity(
            Xc, txt[c].unsqueeze(0).expand(Xc.shape[0], -1), dim=-1
        ).numpy()
        cls_cos_stats[c] = {
            "mean": float(bank_cos_c.mean()),
            "std": float(bank_cos_c.std() + 1e-8),
            "sorted_vals": np.sort(bank_cos_c),
        }

    if correct_only_manifold:
        print(f"  [ICS] correct-only manifold fallback classes: {len(fallback_classes)}/{n_classes}")

    iss = torch.zeros(n_classes)
    for c in range(n_classes):
        smu = cls_sub_mu[c]
        ds = [(smu[i] - smu[j]).norm().item() for i in range(n_sub) for j in range(i + 1, n_sub)]
        iss[c] = float(np.mean(ds)) if ds else 0.0

    cls_mf = torch.cat([
        cls_sigma.unsqueeze(1), cls_deff.unsqueeze(1), cls_sub_sig, cls_sub_pi, iss.unsqueeze(1)
    ], dim=1)
    return {
        "cls_mf": cls_mf, "cls_mu": cls_mu, "cls_var_diag": cls_var_diag,
        "cls_sub_mu": cls_sub_mu, "cls_sub_sig": cls_sub_sig,
        "cls_fvec": cls_fvec, "cls_knn": cls_knn, "cls_cos_stats": cls_cos_stats,
        "feat_dim_cls": cls_mf.shape[1], "feat_dim_inst": n_sub + 10,
        "n_sub": n_sub, "n_neighbors": int(CFG.ics_n_neighbors),
        "text_features": txt, "correct_only_manifold": bool(correct_only_manifold),
        "fallback_classes": fallback_classes,
    }


def compute_ics_instance_features(vis, hyp_labels, mc):
    vis_t = vis.float().cpu() if isinstance(vis, torch.Tensor) else torch.from_numpy(np.asarray(vis)).float()
    hyp = np.asarray(hyp_labels, dtype=np.int64)
    txt = mc["text_features"]
    n_classes = txt.shape[0]
    n_sub = mc["n_sub"]
    out = torch.full((len(vis_t), n_sub + 10), float("nan"))

    for c in range(n_classes):
        idx_c = np.where(hyp == c)[0]
        if len(idx_c) == 0:
            continue
        mu_c = mc["cls_mu"][c]
        var_c = mc["cls_var_diag"][c]
        smu_c = mc["cls_sub_mu"][c]
        ssig_c = mc["cls_sub_sig"][c]
        txt_c = txt[c]
        cos_stats = mc.get("cls_cos_stats", {}).get(c)
        kfn = mc["cls_knn"].get(c)
        fv = mc["cls_fvec"].get(c)
        nfc = 0 if fv is None else fv.shape[0]
        ku = min(mc["n_neighbors"], max(nfc - 1, 1))
        ad_ref = ((fv - mu_c).norm(dim=-1).numpy() if fv is not None and nfc > 0 else np.array([0.0]))

        for gi in idx_c:
            xi = vis_t[gi]
            dm = float((xi - mu_c).norm())
            dma = float(((xi - mu_c).pow(2) / var_c.clamp(1e-6)).sum().sqrt())
            if kfn and nfc > 1:
                dn, idn = kfn(xi.numpy(), ku)
                rho = float(dn.mean()) if len(dn) > 0 else 0.0
                cn = float(F.cosine_similarity(xi.unsqueeze(0).expand(len(idn), -1), fv[idn], dim=-1).mean()) if len(idn) > 0 else 0.0
            else:
                rho, cn, idn = 0.0, 0.0, np.array([], dtype=np.int64)

            sd = torch.stack([(xi - smu_c[j]).norm() / (ssig_c[j] + 1e-6) for j in range(n_sub)])
            aff = F.softmax(-sd, dim=0)
            ha = float(-(aff * (aff + 1e-9).log()).sum())
            bs = int(aff.argmax())
            if fv is not None and len(idn) > 0:
                nbrs = fv[idn]
                if nbrs.dim() == 1:
                    nbrs = nbrs.unsqueeze(0)
                dists = torch.stack([(nbrs - smu_c[j]).norm(dim=-1) for j in range(n_sub)], dim=1)
                pu = float((dists.argmin(dim=1) == bs).float().mean())
            else:
                pu = 0.0
            clip_cos = float(F.cosine_similarity(xi.unsqueeze(0), txt_c.unsqueeze(0)))
            pct_rank = float((ad_ref <= dm).mean())
            if cos_stats is not None:
                intra_cos_zscore = (clip_cos - cos_stats["mean"]) / cos_stats["std"]
                sorted_vals = cos_stats["sorted_vals"]
                intra_cos_pctile = float(np.searchsorted(sorted_vals, clip_cos) / max(len(sorted_vals), 1))
            else:
                intra_cos_zscore = 0.0
                intra_cos_pctile = 0.5
            out[gi] = torch.tensor([
                dm, dma, rho, cn, *aff.tolist(), ha, pu, pct_rank, clip_cos,
                intra_cos_zscore, intra_cos_pctile
            ])

    nan_mask = out.isnan().any(dim=1)
    if nan_mask.any():
        print(f"  [ICS] WARNING: {int(nan_mask.sum())} samples had no matching class; zero-filled.")
        out[nan_mask] = 0.0
    return out


@torch.no_grad()
def score_ics_model(model, vis, txt, inst, cls_feat, batch_size=512):
    model.eval()
    out = []
    for i in range(0, len(vis), batch_size):
        sl = slice(i, min(i + batch_size, len(vis)))
        lo, _, _ = model(
            vis[sl].float().to(DEVICE),
            txt[sl].float().to(DEVICE),
            inst[sl].float().to(DEVICE),
            cls_feat[sl].float().to(DEVICE),
        )
        out.append(torch.sigmoid(lo).cpu())
    return torch.cat(out).numpy().ravel()


def train_one_ics_setting(encoded, text_features, mc, seed, lr, batch_size, verbose=True, train_conditioning="true", target_context=None):
    set_seed(seed)
    txt_all = text_features.float().cpu()
    train_v, train_y = encoded["train"]["v"], encoded["train"]["y"]
    val_v, val_y = encoded["val"]["v"], encoded["val"]["y"]
    train_logits = (np.asarray(target_context["train_logits"], dtype=np.float32) if target_context is not None else _clip_split_logits(train_v, txt_all))
    val_logits = (np.asarray(target_context["val_logits"], dtype=np.float32) if target_context is not None else _clip_split_logits(val_v, txt_all))
    clip_train_preds = train_logits.argmax(axis=1).astype(np.int64)
    clip_val_preds = val_logits.argmax(axis=1).astype(np.int64)
    train_preds = (np.asarray(target_context["train_preds"], dtype=np.int64) if target_context is not None else clip_train_preds)
    val_preds = (np.asarray(target_context["val_preds"], dtype=np.int64) if target_context is not None else clip_val_preds)
    train_fail = (np.asarray(target_context["train_failure"], dtype=np.float32) if target_context is not None else _failure_labels_from_logits(train_logits, train_y))

    train_true = _np(train_y).astype(np.int64)
    train_hyp = train_preds if str(train_conditioning).lower().startswith("pred") else train_true
    train_inst = compute_ics_instance_features(train_v, train_hyp, mc)
    train_cls = mc["cls_mf"][train_hyp]
    train_txt = txt_all[train_hyp]
    val_inst = compute_ics_instance_features(val_v, val_preds, mc)
    val_cls = mc["cls_mf"][val_preds]
    val_txt = txt_all[val_preds]

    model = ICSModel(train_v.shape[1], mc["feat_dim_inst"], mc["feat_dim_cls"], mc["n_sub"], CFG.ics_freeze_iters).to(DEVICE)
    pos_w = torch.tensor([(len(train_fail) - train_fail.sum()) / max(train_fail.sum(), 1.0)], device=DEVICE)
    ds = TensorDataset(train_v.float(), train_txt.float(), train_inst.float(), train_cls.float(), torch.from_numpy(train_fail).float().unsqueeze(1))
    sampler = UniversalBalancedSampler(
        _np(train_y).astype(np.int64), train_fail, batch_size=int(batch_size),
        fail_ratio=CFG.ics_fail_batch_ratio, seed=seed,
    )
    loader = DataLoader(ds, batch_sampler=sampler, num_workers=0)
    proj_p = list(model.in_proj_v.parameters()) + list(model.in_proj_t.parameters())
    other_p = [p for n, p in model.named_parameters() if "in_proj" not in n]
    opt = optim.AdamW([
        {"params": other_p, "lr": float(lr), "lr_scale": 1.0, "weight_decay": CFG.ics_weight_decay},
        {"params": proj_p, "lr": float(lr) * 0.3, "lr_scale": 0.3, "weight_decay": CFG.ics_weight_decay},
    ], betas=(0.9, 0.999))
    scaler = GradScaler("cuda", enabled=USE_AMP)

    best_auc, best_fpr, best_epoch, bad = -np.inf, np.inf, 0, 0
    best_state = copy.deepcopy(model.state_dict())
    history = []
    for epoch in range(CFG.max_epochs):
        model.train()
        epoch_lr = cosine_warmup_lr(float(lr), epoch, CFG.max_epochs, warmup=min(5, max(1, CFG.max_epochs // 5)))
        set_lr(opt, epoch_lr)
        losses = []
        for vb, tb, ib, cb, fb in loader:
            vb, tb, ib, cb, fb = [x.to(DEVICE) for x in (vb, tb, ib, cb, fb)]
            opt.zero_grad(set_to_none=True)
            with autocast("cuda", enabled=USE_AMP):
                lo, cap_p, cap_t = model(vb, tb, ib, cb)
                loss = F.binary_cross_entropy_with_logits(lo, fb, pos_weight=pos_w)
                loss = loss + CFG.caption_aux_weight * (1 - (cap_p * cap_t).sum(-1)).mean()
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), CFG.ics_grad_clip)
            scaler.step(opt)
            scaler.update()
            losses.append(float(loss.detach().cpu()))
        val_scores = score_ics_model(model, val_v, val_txt, val_inst, val_cls)
        val_result = evaluate_failure_scores(val_scores, val_y, val_v, txt_all, CFG.logit_scale, correct_mask=None if target_context is None else target_context["val_correct"], target_name="CLIP" if target_context is None else target_context["name"])
        auc, fpr = float(val_result["auc"]), float(val_result["fpr95"])
        history.append({"epoch": epoch + 1, "loss": float(np.mean(losses)), "val_auc": auc, "val_fpr95": fpr})
        if np.isfinite(auc) and (auc > best_auc + 1e-12 or (abs(auc - best_auc) <= 1e-12 and fpr < best_fpr)):
            best_auc, best_fpr, best_epoch = auc, fpr, epoch + 1
            best_state = copy.deepcopy(model.state_dict())
            bad = 0
        else:
            bad += 1
        if verbose and ((epoch + 1) % 5 == 0 or epoch == 0):
            print(f"      epoch={epoch+1:03d} loss={history[-1]['loss']:.4f} val_auc={auc*100:.2f}% val_fpr95={fpr*100:.2f}%")
        if bad >= CFG.patience:
            break
    model.load_state_dict(best_state)
    return model, {
        "lr": float(lr), "batch_size": int(batch_size), "best_epoch": int(best_epoch),
        "val_auc": float(best_auc), "val_fpr95": float(best_fpr),
        "history": history, "train_conditioning": str(train_conditioning),
    }


# ─── CANONICAL ICS CONFIG (no variants) ───
ICS_CANONICAL_CONFIG = {
    "name": "ICS",
    "train_conditioning": "true",
    "correct_only_manifold": False,
    "description": "canonical ICS: train true-label conditioning, all true-label manifold",
}


def run_ics_model(encoded, text_features, seed, fixed_selection=None, variant_name="ICS",
                  train_conditioning="true", correct_only_manifold=False, target_context=None):
    mode = "validation-only LR/batch-size grid search" if fixed_selection is None else "fixed LR/batch-size from first grid-search seed"
    print(f"\n  [{variant_name}] canonical ICS {mode} | "
          f"train_conditioning={train_conditioning} | correct_only_manifold={correct_only_manifold}")
    txt_all = text_features.float().cpu()
    train_logits = (np.asarray(target_context["train_logits"], dtype=np.float32) if target_context is not None else _clip_split_logits(encoded["train"]["v"], txt_all))
    clip_train_preds = train_logits.argmax(axis=1).astype(np.int64)
    train_preds = (np.asarray(target_context["train_preds"], dtype=np.int64) if target_context is not None else clip_train_preds)
    mc = build_ics_manifold(
        encoded["train"]["v"], encoded["train"]["y"], text_features, seed,
        train_logits=train_logits, train_preds_override=train_preds,
        correct_only_manifold=correct_only_manifold, variant_name=variant_name,
    )

    if fixed_selection is None:
        best = None
        best_model = None
        search_results = []
        for lr in CFG.lr_grid:
            for batch_size in CFG.batch_size_grid:
                print(f"    candidate lr={lr:.0e}, batch_size={batch_size}")
                model, info = train_one_ics_setting(
                    encoded, text_features, mc, seed, lr, batch_size,
                    verbose=False, train_conditioning=train_conditioning, target_context=target_context,
                )
                search_results.append(info)
                print(f"      validation AUC={info['val_auc']*100:.2f}% | FPR95={info['val_fpr95']*100:.2f}% | best_epoch={info['best_epoch']}")
                if best is None or info["val_auc"] > best["val_auc"] + 1e-12 or (abs(info["val_auc"] - best["val_auc"]) <= 1e-12 and info["val_fpr95"] < best["val_fpr95"]):
                    if best_model is not None:
                        del best_model
                    best = info
                    best_model = model
                else:
                    del model
                clean_device_cache()
        print(f"    selected lr={best['lr']:.0e}, batch_size={best['batch_size']} (val AUC={best['val_auc']*100:.2f}%, FPR95={best['val_fpr95']*100:.2f}%)")
    else:
        best = {
            "lr": float(fixed_selection["lr"]), "batch_size": int(fixed_selection["batch_size"]),
            "val_auc": float(fixed_selection.get("val_auc", float("nan"))),
            "val_fpr95": float(fixed_selection.get("val_fpr95", float("nan"))),
            "best_epoch": int(fixed_selection.get("best_epoch", 0)),
            "reused_from_grid_seed": fixed_selection.get("grid_seed"),
            "train_conditioning": str(train_conditioning),
        }
        search_results = []
        print(f"    reusing selected lr={best['lr']:.0e}, batch_size={best['batch_size']} from the first grid-search seed")
        best_model, final_info = train_one_ics_setting(
            encoded, text_features, mc, seed, best["lr"], best["batch_size"],
            verbose=False, train_conditioning=train_conditioning, target_context=target_context,
        )
        best = {**best, **{k: final_info[k] for k in ("best_epoch", "val_auc", "val_fpr95")}}

    train_inst = compute_ics_instance_features(encoded["train"]["v"], train_preds, mc)
    train_cls = mc["cls_mf"][train_preds]
    train_txt = txt_all[train_preds]
    val_logits = (np.asarray(target_context["val_logits"], dtype=np.float32) if target_context is not None else _clip_split_logits(encoded["val"]["v"], txt_all))
    test_logits = (np.asarray(target_context["test_logits"], dtype=np.float32) if target_context is not None else _clip_split_logits(encoded["test"]["v"], txt_all))
    clip_val_preds = val_logits.argmax(axis=1).astype(np.int64)
    clip_test_preds = test_logits.argmax(axis=1).astype(np.int64)
    val_preds = (np.asarray(target_context["val_preds"], dtype=np.int64) if target_context is not None else clip_val_preds)
    test_preds = (np.asarray(target_context["test_preds"], dtype=np.int64) if target_context is not None else clip_test_preds)
    val_inst = compute_ics_instance_features(encoded["val"]["v"], val_preds, mc)
    val_cls = mc["cls_mf"][val_preds]
    val_txt = txt_all[val_preds]
    test_inst = compute_ics_instance_features(encoded["test"]["v"], test_preds, mc)
    test_cls = mc["cls_mf"][test_preds]
    test_txt = txt_all[test_preds]
    train_scores = score_ics_model(best_model, encoded["train"]["v"], train_txt, train_inst, train_cls)
    val_scores = score_ics_model(best_model, encoded["val"]["v"], val_txt, val_inst, val_cls)
    test_scores = score_ics_model(best_model, encoded["test"]["v"], test_txt, test_inst, test_cls)
    train_result = evaluate_failure_scores(train_scores, encoded["train"]["y"], encoded["train"]["v"], txt_all, CFG.logit_scale, correct_mask=None if target_context is None else target_context["train_correct"], target_name="CLIP" if target_context is None else target_context["name"])
    val_result = evaluate_failure_scores(val_scores, encoded["val"]["y"], encoded["val"]["v"], txt_all, CFG.logit_scale, correct_mask=None if target_context is None else target_context["val_correct"], target_name="CLIP" if target_context is None else target_context["name"])
    result = evaluate_failure_scores(test_scores, encoded["test"]["y"], encoded["test"]["v"], txt_all, CFG.logit_scale, correct_mask=None if target_context is None else target_context["test_correct"], target_name="CLIP" if target_context is None else target_context["name"])
    variant_meta = {
        "variant_name": variant_name, "train_conditioning": str(train_conditioning),
        "correct_only_manifold": bool(correct_only_manifold),
        "fallback_classes": len(mc.get("fallback_classes", [])),
    }
    result["selected_hp"] = {**variant_meta, "selected": {k: best.get(k) for k in ("lr", "batch_size", "best_epoch")}}
    train_result["selected_hp"] = result["selected_hp"]
    val_result["selected_hp"] = result["selected_hp"]
    del best_model
    clean_device_cache()
    return result, val_result, train_result, {
        "selected": {**best, **variant_meta},
        "final_validation": {**best, **variant_meta},
        "grid": search_results,
    }


def _clip_split_logits(v, txt):
    return (CFG.logit_scale * v @ txt.T).cpu().numpy()


def _failure_labels_from_logits(logits, labels):
    return (logits.argmax(axis=1) != _np(labels)).astype(np.float32)


def _metric_tuple(result):
    return float(result["auc"]), -float(result["fpr95"])


def _fit_fixed_calibrator_on_val(X_train, y_train, X_val, val_y, val_v, text_features, seed, max_iter=4000, val_correct_mask=None, target_name="CLIP"):
    best = None
    for C in (0.03, 0.10, 0.30, 1.00, 3.00):
        cal = make_pipeline(
            StandardScaler(),
            LogisticRegression(C=C, class_weight="balanced", random_state=seed, max_iter=max_iter, solver="lbfgs"),
        )
        cal.fit(X_train, y_train)
        val_scores = cal.predict_proba(X_val)[:, 1].astype(np.float32)
        val_result = evaluate_failure_scores(val_scores, val_y, val_v, text_features, CFG.logit_scale, correct_mask=val_correct_mask, target_name=target_name)
        print(f"      calibrator C={C:g}: val AUC={val_result['auc']*100:.2f}% FPR@95={val_result['fpr95']*100:.2f}%")
        item = {"calibrator": cal, "C": C, "val": val_result}
        if best is None or _metric_tuple(val_result) > _metric_tuple(best["val"]):
            best = item
    return best


def _require_skopt():
    if not SKOPT_AVAILABLE:
        raise ImportError("scikit-optimize is required for BO search. Install it with: pip install scikit-optimize")


def _bo_minimize(name, dimensions, objective, n_calls, seed, n_initial_points=12):
    _require_skopt()
    n_calls = int(max(1, n_calls))
    n_initial_points = int(min(max(1, n_initial_points), n_calls))
    print(f"  [{name} BO] calls={n_calls}, initial_random={n_initial_points}")
    return gp_minimize(
        objective, dimensions, n_calls=n_calls, n_initial_points=n_initial_points,
        acq_func="EI", random_state=int(seed), noise=1e-10,
    )


def _bo_loss(result):
    return -float(result["auc"]) + 1e-3 * float(result["fpr95"])


def _decode_swnc_bo(x):
    ks = tuple(sorted(set([2] + [int(np.clip(round(k), 2, CFG.swnc_max_k)) for k in x[:6]])))
    return {"ks": ks, "temp": float(x[6]), "txt_blend": float(x[7]), "assignment_temp": float(x[8])}


def _swnc_bo_dimensions(k_cap):
    k_cap = int(max(2, k_cap))
    return [
        Integer(2, k_cap, name="k1"), Integer(2, k_cap, name="k2"),
        Integer(2, k_cap, name="k3"), Integer(2, k_cap, name="k4"),
        Integer(2, k_cap, name="k5"), Integer(2, k_cap, name="k6"),
        Real(3.0, 50.0, prior="log-uniform", name="neighbor_temp"),
        Real(0.0, 1.0, name="txt_blend"),
        Real(30.0, 150.0, prior="log-uniform", name="assignment_temp"),
    ]


def auglikelihoodqda_sweep_configs():
    return [
        {"pca_dim": 32,  "whiten": False, "correct_only": True,  "shrink_alpha": 0.05},
        {"pca_dim": 48,  "whiten": False, "correct_only": True,  "shrink_alpha": 0.05},
        {"pca_dim": 64,  "whiten": False, "correct_only": True,  "shrink_alpha": 0.05},
        {"pca_dim": 96,  "whiten": False, "correct_only": True,  "shrink_alpha": 0.05},
        {"pca_dim": 128, "whiten": False, "correct_only": True,  "shrink_alpha": 0.05},
        {"pca_dim": 64,  "whiten": False, "correct_only": False, "shrink_alpha": 0.05},
        {"pca_dim": 96,  "whiten": False, "correct_only": False, "shrink_alpha": 0.05},
        {"pca_dim": 64,  "whiten": True,  "correct_only": True,  "shrink_alpha": 0.05},
        {"pca_dim": 96,  "whiten": True,  "correct_only": True,  "shrink_alpha": 0.05},
        {"pca_dim": 64,  "whiten": False, "correct_only": True,  "shrink_alpha": 0.00},
        {"pca_dim": 64,  "whiten": False, "correct_only": True,  "shrink_alpha": 0.10},
    ]


def build_class_prototypes(bank_vis_np, bank_labels_np, n_classes):
    protos = []
    for c in range(n_classes):
        mask = bank_labels_np == c
        if mask.sum() == 0:
            protos.append(np.zeros(bank_vis_np.shape[1], dtype=np.float32))
            continue
        mu = bank_vis_np[mask].mean(axis=0)
        protos.append((mu / (np.linalg.norm(mu) + 1e-12)).astype(np.float32))
    return np.stack(protos, axis=0)


def swnc_feature_matrix_v3(query_vis, query_logits, bank_vis_np, bank_labels_np, proto_np, txt_np,
                           ks=(2, 5, 12), temp=15.0, txt_blend=0.70, assignment_temp=100.0,
                           query_bank_indices=None, desc="scoring", pred_labels=None):
    query_np = _np(query_vis).astype(np.float32)
    preds_np = np.argmax(_np(query_logits), axis=1) if pred_labels is None else np.asarray(pred_labels, dtype=np.int64)
    n_q = len(query_np)
    n_classes = proto_np.shape[0]
    ks = tuple(sorted(set([2] + [int(k) for k in ks])))
    temp = float(temp)
    alpha = float(txt_blend)
    assignment_temp = float(assignment_temp)
    max_k = max(ks)

    txt_aug = alpha * txt_np + (1.0 - alpha) * proto_np
    txt_aug = txt_aug / (np.linalg.norm(txt_aug, axis=1, keepdims=True) + 1e-12)

    sim_q_txt = query_np @ txt_aug.T
    assignment = scipy_softmax(sim_q_txt * assignment_temp, axis=1)
    semantic_query = assignment @ txt_aug
    semantic_query = semantic_query / (np.linalg.norm(semantic_query, axis=1, keepdims=True) + 1e-12)

    n_per_view = len(ks) * 6 + 8
    n_cross = 7
    feats = np.zeros((n_q, n_per_view * 2 + n_cross), dtype=np.float32)

    for start_i in range(0, n_q, 512):
        end_i = min(start_i + 512, n_q)
        q_batch = query_np[start_i:end_i]
        s_batch = semantic_query[start_i:end_i]
        sims_vis = q_batch @ bank_vis_np.T
        sims_sem = s_batch @ bank_vis_np.T

        if query_bank_indices is not None:
            for i, idx in enumerate(query_bank_indices[start_i:end_i]):
                if 0 <= int(idx) < sims_vis.shape[1]:
                    sims_vis[i, int(idx)] = -1e9
                    sims_sem[i, int(idx)] = -1e9

        proto_sims_vis = q_batch @ proto_np.T
        proto_sims_sem = s_batch @ proto_np.T

        for i in range(end_i - start_i):
            gi = start_i + i
            pred = int(preds_np[gi])

            def extract_view_feats(sim_row, proto_row):
                kk = min(max_k, len(sim_row))
                idx = np.argpartition(-sim_row, kk - 1)[:kk]
                idx = idx[np.argsort(-sim_row[idx])]
                s_sorted = sim_row[idx]
                l_sorted = bank_labels_np[idx]

                f = []
                for k in ks:
                    k_e = min(k, kk)
                    sk, lk = s_sorted[:k_e], l_sorted[:k_e]
                    wk = np.exp((sk - sk.max()) / (temp + 1e-12))
                    wk /= wk.sum() + 1e-12
                    counts = np.bincount(lk.astype(np.int64), weights=wk, minlength=n_classes)
                    pm = counts[pred]
                    others = np.delete(counts, pred) if n_classes > 1 else np.array([0.0])
                    second = np.sort(counts)[::-1][1] if len(counts) > 1 else 0.0
                    margin = pm - others.max()
                    hard_miss = 1.0 - (lk == pred).mean()
                    f.extend([
                        1.0 - pm,
                        float(-(counts * np.log(counts + 1e-12)).sum()),
                        float(-margin), float(hard_miss), float(second),
                        float(np.log1p(pm / (second + 1e-8) if second > 0 else pm)),
                    ])

                top5 = s_sorted[:5].mean() if len(s_sorted) >= 5 else s_sorted.mean()
                top10 = s_sorted[:10].mean() if len(s_sorted) >= 10 else s_sorted.mean()
                top20 = s_sorted[:20].mean() if len(s_sorted) >= 20 else s_sorted.mean()
                sim_drop = s_sorted[0] - s_sorted[1] if kk > 1 else 0.0
                sim_gap = s_sorted[0] - s_sorted.mean() if kk > 1 else 0.0
                sim_std = s_sorted[:5].std() if len(s_sorted) >= 5 else 0.0
                f.extend([
                    float(proto_row[pred]), float(proto_row.max() - proto_row[pred]),
                    float(top5), float(top10), float(top20),
                    float(sim_drop), float(sim_gap), float(sim_std),
                ])
                return f, idx[:10], l_sorted[:10]

            f_v, idx_v, lbl_v = extract_view_feats(sims_vis[i], proto_sims_vis[i])
            f_s, idx_s, lbl_s = extract_view_feats(sims_sem[i], proto_sims_sem[i])
            vis_agree = float((lbl_v == pred).mean()) if len(lbl_v) else 0.0
            sem_agree = float((lbl_s == pred).mean()) if len(lbl_s) else 0.0
            union = set(idx_v) | set(idx_s)
            f_cross = [
                len(set(idx_v) & set(idx_s)) / max(len(union), 1),
                float((lbl_v == lbl_s).mean()) if len(lbl_v) and len(lbl_s) else 0.0,
                vis_agree, sem_agree,
                float(np.dot(q_batch[i], s_batch[i])),
                abs(vis_agree - sem_agree), min(vis_agree, sem_agree),
            ]
            feats[gi] = np.array(f_v + f_s + f_cross, dtype=np.float32)
    return feats


def run_classical_models(encoded, text_features, seed, fixed_selection=None, target_context=None):
    """AugLikelihoodQDA/SWNC on fixed train/val/test. Search once, then reuse selected configs across seeds."""
    txt = text_features.float().cpu() if isinstance(text_features, torch.Tensor) else torch.from_numpy(text_features).float()
    train_v, train_y = encoded["train"]["v"], encoded["train"]["y"]
    val_v, val_y = encoded["val"]["v"], encoded["val"]["y"]
    test_v, test_y = encoded["test"]["v"], encoded["test"]["y"]
    train_logits = (np.asarray(target_context["train_logits"], dtype=np.float32) if target_context is not None else _clip_split_logits(train_v, txt))
    val_logits = (np.asarray(target_context["val_logits"], dtype=np.float32) if target_context is not None else _clip_split_logits(val_v, txt))
    test_logits = (np.asarray(target_context["test_logits"], dtype=np.float32) if target_context is not None else _clip_split_logits(test_v, txt))
    train_fail = (np.asarray(target_context["train_failure"], dtype=np.float32) if target_context is not None else _failure_labels_from_logits(train_logits, train_y))
    train_preds = (np.asarray(target_context["train_preds"], dtype=np.int64) if target_context is not None else train_logits.argmax(axis=1))
    val_preds = (np.asarray(target_context["val_preds"], dtype=np.int64) if target_context is not None else val_logits.argmax(axis=1))
    test_preds = (np.asarray(target_context["test_preds"], dtype=np.int64) if target_context is not None else test_logits.argmax(axis=1))
    val_correct = None if target_context is None else target_context["val_correct"]
    test_correct = None if target_context is None else target_context["test_correct"]
    train_correct = None if target_context is None else target_context["train_correct"]
    target_name = "CLIP" if target_context is None else target_context["name"]

    print(f"\n{'='*60}")
    print("[CLASSICAL MODELS - SEARCH ON FIRST SEED, REUSE HP ACROSS SEEDS]")
    print(f"  Train calibrator split: {len(train_v)} samples ({int(train_fail.sum())} failures, {100*train_fail.mean():.1f}%)")
    print(f"  Val selection split: {len(val_v)} samples | Test: {len(test_v)} samples")
    print(f"{'='*60}")

    results = {}
    selection = {"AugLikelihoodQDA": {}, "SWNC": {}} if fixed_selection is None else copy.deepcopy(fixed_selection)

    print(f"\n{'-'*60}")
    print("AugLikelihoodQDA top-config sweep" if fixed_selection is None else "AugLikelihoodQDA fixed top-config evaluation")
    print(f"{'-'*60}")
    if fixed_selection is None or not selection.get("AugLikelihoodQDA", {}).get("top_configs"):
        ranked = []
        for cfg in auglikelihoodqda_sweep_configs():
            model = AugLikelihoodQDA(seed=seed, reg=1e-3, calibrator_C=1.0, **cfg).fit(train_v, train_logits, train_y, None)
            raw_val = model.raw_matrix(val_v, val_logits, pred_labels=val_preds)
            val_labels = (~np.asarray(evaluate_failure_scores(raw_val[:, 0], val_y, val_v, txt, CFG.logit_scale, correct_mask=val_correct, target_name=target_name)["correct"])).astype(int)
            aucs = [roc_auc_score(val_labels, raw_val[:, j]) for j in range(raw_val.shape[1])]
            best_auc = max(aucs)
            ranked.append((best_auc, cfg))
            print(f"  {cfg} val best raw-column AUC={best_auc*100:.2f}%")
        ranked.sort(key=lambda x: x[0], reverse=True)
        selection["AugLikelihoodQDA"] = {"top_configs": [cfg for _, cfg in ranked[:3]]}

    qda_models = [AugLikelihoodQDA(seed=seed, reg=1e-3, calibrator_C=1.0, **cfg).fit(train_v, train_logits, train_y, None)
                  for cfg in selection["AugLikelihoodQDA"]["top_configs"]]
    print("  Selected AugLikelihoodQDA raw configs:")
    for cfg in selection["AugLikelihoodQDA"]["top_configs"]:
        print(f"    {cfg}")
    X_tr = np.concatenate([m.raw_matrix(train_v, train_logits, pred_labels=train_preds) for m in qda_models], axis=1)
    X_val = np.concatenate([m.raw_matrix(val_v, val_logits, pred_labels=val_preds) for m in qda_models], axis=1)
    X_te = np.concatenate([m.raw_matrix(test_v, test_logits, pred_labels=test_preds) for m in qda_models], axis=1)
    qda_cal = _fit_fixed_calibrator_on_val(X_tr, train_fail, X_val, val_y, val_v, txt, seed, max_iter=4000, val_correct_mask=val_correct, target_name=target_name)
    qda_train_scores = qda_cal["calibrator"].predict_proba(X_tr)[:, 1].astype(np.float32)
    qda_val_scores = qda_cal["calibrator"].predict_proba(X_val)[:, 1].astype(np.float32)
    qda_test_scores = qda_cal["calibrator"].predict_proba(X_te)[:, 1].astype(np.float32)
    qda_train = evaluate_failure_scores(qda_train_scores, train_y, train_v, txt, CFG.logit_scale, correct_mask=train_correct, target_name=target_name)
    qda_val = evaluate_failure_scores(qda_val_scores, val_y, val_v, txt, CFG.logit_scale, correct_mask=val_correct, target_name=target_name)
    qda_test = evaluate_failure_scores(qda_test_scores, test_y, test_v, txt, CFG.logit_scale, correct_mask=test_correct, target_name=target_name)
    qda_test["selected_hp"] = {"top_configs": selection["AugLikelihoodQDA"]["top_configs"], "calibrator_C": qda_cal["C"], "source": "auglikelihoodqda_sweep"}
    results["AugLikelihoodQDA"] = {"train": qda_train, "val": qda_val, "test": qda_test}
    print(f"  [AugLikelihoodQDA TEST] AUC={qda_test['auc']*100:.2f}% FPR@95={qda_test['fpr95']*100:.2f}% Cohen d={qda_test['cohen_d']:.3f}")

    print(f"\n{'-'*60}")
    print("SWNC Bayesian optimization" if fixed_selection is None else "SWNC fixed-HP evaluation")
    print(f"{'-'*60}")
    bank_vis_np = _np(train_v).astype(np.float32)
    bank_labels_np = _np(train_y).astype(np.int64)
    proto_np = build_class_prototypes(bank_vis_np, bank_labels_np, txt.shape[0])
    txt_np = _np(txt).astype(np.float32)
    k_cap = int(min(CFG.swnc_max_k, max(2, len(bank_vis_np) - 1)))

    if fixed_selection is None or not selection.get("SWNC", {}).get("hp"):
        swnc_trials = []
        def swnc_objective(x):
            hp = _decode_swnc_bo(x)
            hp["ks"] = tuple(k for k in hp["ks"] if k <= k_cap) or (2,)
            print(f"  SWNC BO hp=ks{hp['ks']}, temp={hp['temp']:.3g}, txt_blend={hp['txt_blend']:.3g}, assignment_temp={hp['assignment_temp']:.3g}")
            X_tr = swnc_feature_matrix_v3(train_v, train_logits, bank_vis_np, bank_labels_np, proto_np, txt_np,
                                          ks=hp["ks"], temp=hp["temp"], txt_blend=hp["txt_blend"], assignment_temp=hp["assignment_temp"],
                                          query_bank_indices=np.arange(len(train_v)), desc=f"train ks={hp['ks']}", pred_labels=train_preds)
            X_val = swnc_feature_matrix_v3(val_v, val_logits, bank_vis_np, bank_labels_np, proto_np, txt_np,
                                           ks=hp["ks"], temp=hp["temp"], txt_blend=hp["txt_blend"], assignment_temp=hp["assignment_temp"],
                                           query_bank_indices=None, desc=f"val ks={hp['ks']}", pred_labels=val_preds)
            keep = X_tr.std(axis=0) > 1e-6
            cal = make_pipeline(StandardScaler(), LogisticRegressionCV(Cs=20, cv=3, scoring="roc_auc", class_weight="balanced", random_state=seed, max_iter=4000, solver="lbfgs"))
            cal.fit(X_tr[:, keep], train_fail)
            val_scores = cal.predict_proba(X_val[:, keep])[:, 1].astype(np.float32)
            val_result = evaluate_failure_scores(val_scores, val_y, val_v, txt, CFG.logit_scale, correct_mask=val_correct, target_name=target_name)
            swnc_trials.append({"hp": dict(hp), "keep": keep, "val": val_result})
            print(f"    -> val AUC={val_result['auc']*100:.2f}% FPR@95={val_result['fpr95']*100:.2f}% features={int(keep.sum())}")
            return _bo_loss(val_result)
        _bo_minimize("SWNC", _swnc_bo_dimensions(k_cap), swnc_objective, CFG.swnc_bo_calls, seed + 1009, n_initial_points=18)
        swnc_trials.sort(key=lambda t: _metric_tuple(t["val"]), reverse=True)
        selection["SWNC"] = {"hp": swnc_trials[0]["hp"]}

    hp = selection["SWNC"]["hp"]
    print(f"  [SWNC SELECTED] ks={hp['ks']}, temp={hp['temp']:.3g}, txt_blend={hp['txt_blend']:.3g}, assignment_temp={hp['assignment_temp']:.3g}")
    X_tr = swnc_feature_matrix_v3(train_v, train_logits, bank_vis_np, bank_labels_np, proto_np, txt_np,
                                  ks=hp["ks"], temp=hp["temp"], txt_blend=hp["txt_blend"], assignment_temp=hp["assignment_temp"],
                                  query_bank_indices=np.arange(len(train_v)), desc=f"train-final ks={hp['ks']}", pred_labels=train_preds)
    X_val = swnc_feature_matrix_v3(val_v, val_logits, bank_vis_np, bank_labels_np, proto_np, txt_np,
                                   ks=hp["ks"], temp=hp["temp"], txt_blend=hp["txt_blend"], assignment_temp=hp["assignment_temp"],
                                   query_bank_indices=None, desc=f"val-final ks={hp['ks']}", pred_labels=val_preds)
    X_te = swnc_feature_matrix_v3(test_v, test_logits, bank_vis_np, bank_labels_np, proto_np, txt_np,
                                  ks=hp["ks"], temp=hp["temp"], txt_blend=hp["txt_blend"], assignment_temp=hp["assignment_temp"],
                                  query_bank_indices=None, desc=f"test ks={hp['ks']}", pred_labels=test_preds)
    keep = X_tr.std(axis=0) > 1e-6
    swnc_cal = make_pipeline(StandardScaler(), LogisticRegressionCV(Cs=20, cv=3, scoring="roc_auc", class_weight="balanced", random_state=seed, max_iter=4000, solver="lbfgs"))
    swnc_cal.fit(X_tr[:, keep], train_fail)
    selected_swnc_c = float(swnc_cal.named_steps["logisticregressioncv"].C_[0])
    swnc_train_scores = swnc_cal.predict_proba(X_tr[:, keep])[:, 1].astype(np.float32)
    swnc_val_scores = swnc_cal.predict_proba(X_val[:, keep])[:, 1].astype(np.float32)
    swnc_test_scores = swnc_cal.predict_proba(X_te[:, keep])[:, 1].astype(np.float32)
    swnc_train = evaluate_failure_scores(swnc_train_scores, train_y, train_v, txt, CFG.logit_scale, correct_mask=train_correct, target_name=target_name)
    swnc_val = evaluate_failure_scores(swnc_val_scores, val_y, val_v, txt, CFG.logit_scale, correct_mask=val_correct, target_name=target_name)
    swnc_test = evaluate_failure_scores(swnc_test_scores, test_y, test_v, txt, CFG.logit_scale, correct_mask=test_correct, target_name=target_name)
    swnc_test["selected_hp"] = {"bo_calls": int(CFG.swnc_bo_calls), **hp, "final_calibrator_C": selected_swnc_c, "calibrator": "LogisticRegressionCV(Cs=20, cv=3, scoring='roc_auc')"}
    results["SWNC"] = {"train": swnc_train, "val": swnc_val, "test": swnc_test}
    print(f"  [SWNC TEST] AUC={swnc_test['auc']*100:.2f}% FPR@95={swnc_test['fpr95']*100:.2f}% Cohen d={swnc_test['cohen_d']:.3f}")

    return results, selection


def _score_matrix(score_dict, names):
    return np.column_stack([np.asarray(score_dict[name]["scores"], dtype=float).reshape(-1) for name in names])


def sigmoid_norm(scores, mu, std):
    z = (np.asarray(scores, dtype=float) - float(mu)) / (float(std) + 1e-10)
    z = np.clip(z, -60.0, 60.0)
    return (1.0 / (1.0 + np.exp(-z))).astype(np.float32)


def _fusion_metrics_from_correct(scores, correct):
    scores = np.asarray(scores, dtype=float)
    correct = np.asarray(correct, dtype=bool)
    return {
        "auc": get_auroc(scores[correct], scores[~correct]),
        "fpr95": get_fpr95(scores[correct], scores[~correct]),
    }


def _joint_score(m):
    return float(m["auc"]) - float(m["fpr95"])


def _spearman_corr_df(score_matrix, names):
    rho, _ = spearmanr(np.asarray(score_matrix, dtype=float), axis=0)
    rho = np.asarray(rho, dtype=float)
    if rho.ndim == 0:
        rho = np.array([[1.0]])
    return pd.DataFrame(rho, index=names, columns=names)


def _print_weight_table(title, names, weights):
    names = list(names)
    weights = np.asarray(weights, dtype=float).reshape(-1)
    print(f"    Weights [{title}]:")
    for label, weight in zip(names, weights):
        print(f"      {label:<18}: {float(weight): .4f}")


def _fit_bayes_weights(val_matrix, val_correct, seed):
    n = val_matrix.shape[1]
    def objective(weights):
        w = np.abs(np.asarray(weights, dtype=np.float64))
        s = w.sum()
        if s < 1e-12:
            return 1.0
        w /= s
        m = _fusion_metrics_from_correct(val_matrix @ w, val_correct)
        return float(m["fpr95"] - m["auc"])

    if SKOPT_AVAILABLE:
        res = gp_minimize(
            objective,
            [Real(0.0, 1.0, name=f"w{i}") for i in range(n)],
            n_calls=int(CFG.ensemble_bayes_calls),
            n_initial_points=min(int(CFG.ensemble_bayes_random_starts), int(CFG.ensemble_bayes_calls)),
            random_state=int(seed),
            verbose=False,
        )
        w = np.abs(np.asarray(res.x, dtype=np.float64))
    else:
        rng = np.random.default_rng(int(seed))
        best_obj, w = float("inf"), np.ones(n, dtype=np.float64) / n
        for _ in range(10000):
            cand = rng.dirichlet(np.ones(n))
            obj = objective(cand)
            if obj < best_obj:
                best_obj, w = obj, cand
    return (w / (w.sum() + 1e-12)).astype(np.float64)


def run_ensembles(score_registry, encoded, text_features, seed, target_context=None):
    """Bayesian Fusion over the 5 proposed models only:
       SOARER, AugLikelihoodQDA, ATLAS, SWNC, ICS."""
    proposed_order = ["SOARER", "AugLikelihoodQDA", "ATLAS", "SWNC", "ICS"]
    proposed = [name for name in proposed_order
                if name in score_registry["val"] and name in score_registry["train"]]
    if len(proposed) < 2:
        print("Skipping Bayesian Fusion: fewer than two proposed models available.")
        return {}

    train_correct = np.asarray(next(iter(score_registry["train"].values()))["correct"], dtype=bool)
    val_correct = np.asarray(next(iter(score_registry["val"].values()))["correct"], dtype=bool)
    test_correct = None if target_context is None else target_context["test_correct"]
    target_name = "CLIP" if target_context is None else target_context["name"]

    train_norm, val_norm, test_norm = {}, {}, {}
    for mn in proposed:
        tr = np.asarray(score_registry["train"][mn]["scores"], dtype=float)
        ref = tr[train_correct]
        if len(ref) < 2:
            ref = tr
        mu = float(ref.mean())
        std = float(ref.std()) + 1e-10
        train_norm[mn] = sigmoid_norm(tr, mu, std)
        val_norm[mn] = sigmoid_norm(score_registry["val"][mn]["scores"], mu, std)
        test_norm[mn] = sigmoid_norm(score_registry["test"][mn]["scores"], mu, std)

    V = np.column_stack([val_norm[m] for m in proposed])
    T = np.column_stack([test_norm[m] for m in proposed])

    spearman_df = _spearman_corr_df(V, proposed)
    offdiag = spearman_df.to_numpy()[~np.eye(len(proposed), dtype=bool)]
    mean_abs_spearman = float(np.nanmean(np.abs(offdiag))) if len(offdiag) else float("nan")

    print(f"\n{'='*60}")
    print("BAYESIAN FUSION (proposed models only)")
    print(f"Members: {', '.join(proposed)}")
    print(f"{'='*60}")
    print("  Spearman correlation on normalized validation uncertainty scores:")
    display(spearman_df.style.format("{:.3f}"))
    print(f"  Mean |off-diagonal Spearman|: {mean_abs_spearman:.3f}")
    print("  Individual validation metrics after train-success sigmoid normalization:")
    for mn in proposed:
        vm = _fusion_metrics_from_correct(val_norm[mn], val_correct)
        print(f"    {mn:<18} AUC={vm['auc']*100:>7.2f}% FPR@95={vm['fpr95']*100:>7.2f}%")

    bayes_w = _fit_bayes_weights(V, val_correct, seed + 3001)
    bayes_val_scores = V @ bayes_w
    bayes_test_scores = T @ bayes_w
    bayes_val_m = _fusion_metrics_from_correct(bayes_val_scores, val_correct)

    fused_result = evaluate_failure_scores(
        bayes_test_scores, encoded["test"]["y"], encoded["test"]["v"],
        text_features, CFG.logit_scale, correct_mask=test_correct, target_name=target_name,
    )
    fused_result.update({
        "members": proposed,
        "weights": bayes_w.tolist(),
        "val_auc": bayes_val_m["auc"],
        "val_fpr95": bayes_val_m["fpr95"],
        "fusion_protocol": "train-success sigmoid normalization + Bayesian optimization over simplex weights",
        "mean_abs_spearman": mean_abs_spearman,
        "is_best_validation_fusion": True,
    })

    print(
        f"  Bayes Fusion   val_joint={_joint_score(bayes_val_m):>7.4f} "
        f"val_AUC={fused_result['val_auc']*100:>7.2f}% val_FPR@95={fused_result['val_fpr95']*100:>7.2f}% | "
        f"test_AUC={fused_result['auc']*100:>7.2f}% test_FPR@95={fused_result['fpr95']*100:>7.2f}% "
        f"Cohen d={fused_result['cohen_d']:>7.3f}"
    )
    _print_weight_table("Bayes Fusion", proposed, bayes_w)

    return {"Bayes Fusion": fused_result}


def plot_uncertainty_distributions(score_registry, title_suffix=None):
    import matplotlib.pyplot as plt
    if title_suffix is None:
        title_suffix = f"{CFG.dataset_name} test set"

    names = list(score_registry.keys())
    n = len(names)
    ncols = 3
    nrows = int(math.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5.2 * ncols, 3.8 * nrows), squeeze=False)
    bins = np.linspace(0.0, 1.0, 41)

    for ax, name in zip(axes.ravel(), names):
        result = score_registry[name]
        uncertainty = normalize_uncertainty_scores(result["scores"])
        correct = np.asarray(result["correct"], dtype=bool)
        ax.hist(uncertainty[correct], bins=bins, density=True, alpha=0.55, color="#1f77b4", label="Correct")
        ax.hist(uncertainty[~correct], bins=bins, density=True, alpha=0.55, color="#d62728", label="Failure")
        theta = result.get("youden_theta", np.nan)
        if np.isfinite(theta):
            ax.axvline(theta, color="black", linestyle="--", linewidth=1.5, label=r"Youden $\theta^*$")
        ax.set_title(f"{name} | d={result.get('cohen_d', np.nan):.2f}")
        ax.set_xlabel("Normalized uncertainty score")
        ax.set_ylabel("Density")
        ax.set_xlim(0, 1)
        ax.grid(True, alpha=0.25)
        ax.legend(fontsize=8)

    for ax in axes.ravel()[n:]:
        ax.axis("off")
    fig.suptitle(f"Normalized uncertainty score distributions on the {title_suffix}", fontsize=14)
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show(block=True)
    return fig


def _print_seed_live_summary(target_name, seed, seed_registry, ensemble_results):
    print(f"\n{'*'*72}")
    print(f"  LIVE SEED SUMMARY | Target={target_name} | Seed={seed}")
    print(f"{'*'*72}")
    header = f"  {'Model':<22} {'AUC':>10} {'FPR@95':>10} {'Cohen d':>10}"
    print(header)
    print(f"  {'-'*20:<22} {'-'*8:>10} {'-'*8:>10} {'-'*8:>10}")
    for name, res in seed_registry["test"].items():
        auc = float(res.get("auc", float("nan")))
        fpr = float(res.get("fpr95", float("nan")))
        cd = float(res.get("cohen_d", float("nan")))
        print(f"  {name:<22} {auc*100:>9.2f}% {fpr*100:>9.2f}% {cd:>10.3f}")
    if ensemble_results:
        print(f"  {'-'*20:<22} {'-'*8:>10} {'-'*8:>10} {'-'*8:>10}")
        for name, res in ensemble_results.items():
            auc = float(res.get("auc", float("nan")))
            fpr = float(res.get("fpr95", float("nan")))
            cd = float(res.get("cohen_d", float("nan")))
            print(f"  {name:<22} {auc*100:>9.2f}% {fpr*100:>9.2f}% {cd:>10.3f}  <-- ensemble")
    print(f"{'*'*72}\n")


# ------------------------------ EXECUTION ----------------------------------

bundle = build_dataset_bundle(CFG)
required = {"train", "val", "test", "class_names"}
missing = required.difference(bundle)
if missing:
    raise KeyError(f"Dataset bundle is missing: {sorted(missing)}")

encoded, text_features = load_or_encode_features(bundle)
target_predictions = load_or_compute_target_predictions(bundle, encoded, text_features)

all_rows = []
all_vilu_ablation_rows = []  # kept for CSV schema compatibility; will remain empty
final_plot_registries = {}


def run_experiment_for_target(target_name, target_context):
    target_text_features = target_context["text_features"]
    target_encoded = target_context["encoded"]
    set_active_dims(target_encoded, target_text_features)
    rows = []
    vilu_ablation_rows = []  # unused (no variants)
    score_registry = {"train": {}, "val": {}, "test": {}}
    plot_score_bank = {}

    def add_plot_scores(name, result):
        plot_score_bank.setdefault(name, []).append(np.asarray(result["scores"], dtype=np.float32).reshape(-1))

    def build_final_plot_registry():
        final = {}
        for name, score_blocks in plot_score_bank.items():
            if len(score_blocks) == 1:
                scores = score_blocks[0]
            else:
                scores = np.mean(np.stack(score_blocks, axis=0), axis=0).astype(np.float32)
            result = evaluate_failure_scores(
                scores, target_encoded["test"]["y"], target_encoded["test"]["v"],
                target_text_features, CFG.logit_scale,
                correct_mask=target_context["test_correct"], target_name=target_name,
            )
            result["plot_n_score_sets"] = len(score_blocks)
            final[name] = result
        return final

    # MCM (baseline)
    mcm_train = run_mcm(target_encoded["train"]["v"], target_encoded["train"]["y"], target_text_features, target_context["train_correct"], target_name, target_context["train_preds"], target_context["train_logits"], activation=target_context["activation"])
    mcm_val = run_mcm(target_encoded["val"]["v"], target_encoded["val"]["y"], target_text_features, target_context["val_correct"], target_name, target_context["val_preds"], target_context["val_logits"], activation=target_context["activation"])
    mcm = run_mcm(target_encoded["test"]["v"], target_encoded["test"]["y"], target_text_features, target_context["test_correct"], target_name, target_context["test_preds"], target_context["test_logits"], activation=target_context["activation"])
    print(f"{target_name} zero-shot test accuracy: {mcm['accuracy']*100:.2f}%")
    score_registry["train"]["MCM"] = mcm_train
    score_registry["val"]["MCM"] = mcm_val
    score_registry["test"]["MCM"] = mcm
    add_plot_scores("MCM", mcm)
    rows.append({
        "model": "MCM", "seed": "deterministic", "result_type": "individual",
        **{k: mcm[k] for k in ("auc", "fpr95", "cohen_d", "n_success", "n_failure")}
    })

    # ─── Canonical neural model builders (no variants) ───
    # LVU, ViLU are frozen baselines; SOARER, ATLAS are proposed canonical models.
    neural_builders = {
        "LVU":    build_lvu_model,
        "ViLU":   build_vilu_model,
        "SOARER": build_soarer_model,
        "ATLAS":  build_atlas_model,
    }

    grid_seed = int(CFG.seeds[0]) if len(CFG.seeds) else 42
    selected_hps = {}
    classical_selection = None

    for seed in CFG.seeds:
        print(f"\n{'='*72}\nSEED {seed}\n{'='*72}")
        seed_registry = {"train": {"MCM": mcm_train}, "val": {"MCM": mcm_val}, "test": {"MCM": mcm}}

        # Classical: AugLikelihoodQDA + SWNC (unchanged - no variants here)
        classical, classical_selection = run_classical_models(
            target_encoded, target_text_features, seed, fixed_selection=classical_selection, target_context=target_context
        )
        for name, split_results in classical.items():
            train_result = split_results["train"]
            val_result = split_results["val"]
            result = split_results["test"]
            seed_registry["train"][name] = train_result
            seed_registry["val"][name] = val_result
            seed_registry["test"][name] = result
            score_registry["train"][name] = train_result
            score_registry["val"][name] = val_result
            score_registry["test"][name] = result
            add_plot_scores(name, result)
            rows.append({
                "model": name, "seed": seed, "result_type": "individual",
                "selected_hp": json.dumps(result.get("selected_hp", {})),
                **{k: result[k] for k in ("auc", "fpr95", "cohen_d", "n_success", "n_failure")}
            })
            print(f"{name:<18} AUC={100*result['auc']:.2f}% "
                  f"FPR95={100*result['fpr95']:.2f}% Cohen d={result['cohen_d']:.3f}")

        # Neural models (canonical, no variants): LVU, ViLU, SOARER, ATLAS
        for name, builder in neural_builders.items():
            fixed_selection = selected_hps.get(name)
            result, val_result, train_result, selection = run_neural_model(
                name, builder, target_encoded, target_text_features, seed,
                fixed_selection=fixed_selection, target_context=target_context
            )
            if name not in selected_hps:
                selected_hps[name] = {**selection["selected"], "grid_seed": seed}
            seed_registry["train"][name] = train_result
            seed_registry["val"][name] = val_result
            seed_registry["test"][name] = result
            score_registry["train"][name] = train_result
            score_registry["val"][name] = val_result
            score_registry["test"][name] = result
            add_plot_scores(name, result)
            rows.append({
                "model": name, "seed": seed, "result_type": "individual",
                "selected_lr": selection["selected"]["lr"],
                "selected_batch_size": selection["selected"]["batch_size"],
                "selected_epoch": selection["final_validation"]["best_epoch"],
                "hp_source_seed": selected_hps[name].get("grid_seed"),
                **{k: result[k] for k in ("auc", "fpr95", "cohen_d", "n_success", "n_failure")}
            })
            print(f"{name:<18} AUC={100*result['auc']:.2f}% "
                  f"FPR95={100*result['fpr95']:.2f}% Cohen d={result['cohen_d']:.3f}")

        # ICS (canonical, no variants)
        ics_name = ICS_CANONICAL_CONFIG["name"]
        ics_fixed = selected_hps.get(ics_name)
        ics_test, ics_val, ics_train, ics_selection = run_ics_model(
            target_encoded, target_text_features, seed,
            fixed_selection=ics_fixed, variant_name=ics_name,
            train_conditioning=ICS_CANONICAL_CONFIG["train_conditioning"],
            correct_only_manifold=ICS_CANONICAL_CONFIG["correct_only_manifold"],
            target_context=target_context,
        )
        if ics_name not in selected_hps:
            selected_hps[ics_name] = {**ics_selection["selected"], "grid_seed": seed}
        seed_registry["train"]["ICS"] = ics_train
        seed_registry["val"]["ICS"] = ics_val
        seed_registry["test"]["ICS"] = ics_test
        score_registry["train"]["ICS"] = ics_train
        score_registry["val"]["ICS"] = ics_val
        score_registry["test"]["ICS"] = ics_test
        add_plot_scores("ICS", ics_test)
        rows.append({
            "model": "ICS", "seed": seed, "result_type": "individual",
            "selected_lr": ics_selection["selected"].get("lr"),
            "selected_batch_size": ics_selection["selected"].get("batch_size"),
            "selected_epoch": ics_selection["final_validation"].get("best_epoch"),
            "hp_source_seed": selected_hps[ics_name].get("grid_seed"),
            **{k: ics_test[k] for k in ("auc", "fpr95", "cohen_d", "n_success", "n_failure")}
        })
        print(f"ICS                AUC={100*ics_test['auc']:.2f}% "
              f"FPR95={100*ics_test['fpr95']:.2f}% Cohen d={ics_test['cohen_d']:.3f}")

        # Bayesian Fusion over the 5 proposed models only
        ensemble_results = run_ensembles(seed_registry, target_encoded, target_text_features, seed, target_context=target_context)
        for name, result in ensemble_results.items():
            score_registry["test"][name] = result
            add_plot_scores(name, result)
            rows.append({
                "model": name, "seed": seed, "result_type": "ensemble",
                "members": ", ".join(result.get("members", [])),
                "weights": json.dumps(result.get("weights", [])),
                "val_auc": result.get("val_auc", np.nan),
                "val_fpr95": result.get("val_fpr95", np.nan),
                "is_best_validation_fusion": result.get("is_best_validation_fusion", False),
                "fusion_protocol": result.get("fusion_protocol", ""),
                "mean_abs_spearman": result.get("mean_abs_spearman", np.nan),
                **{k: result[k] for k in ("auc", "fpr95", "cohen_d", "n_success", "n_failure")}
            })

        _print_seed_live_summary(target_name, seed, seed_registry, ensemble_results)

    for row in rows:
        row.setdefault("target_model", target_name)
    return rows, vilu_ablation_rows, build_final_plot_registry()


for target_name in canonical_target_model_names(CFG.zero_shot_target_models):
    if target_name not in target_predictions:
        print(f"Skipping unavailable target model: {target_name}")
        continue
    target_context = make_target_context(target_predictions, target_name)
    print(f"\n{'#'*78}\nTARGET ZERO-SHOT MODEL: {target_name}\n{'#'*78}")
    for split in ("train", "val", "test"):
        acc = float(np.mean(target_context[f"{split}_correct"]))
        print(f"  {split:<5} accuracy: {acc*100:.2f}%")
    target_rows, target_vilu_rows, target_plot_registry = run_experiment_for_target(target_name, target_context)
    all_rows.extend(target_rows)
    all_vilu_ablation_rows.extend(target_vilu_rows)
    final_plot_registries[target_name] = target_plot_registry

rows = all_rows
vilu_ablation_rows = all_vilu_ablation_rows

# ─── Summary + Save ───
raw_results = pd.DataFrame(rows)

def summarize_runs_by_target(raw_df: pd.DataFrame) -> Dict[str, pd.DataFrame]:
    tables = {}
    for target, target_grp in raw_df.groupby("target_model", sort=False):
        tables[target] = summarize_runs(target_grp)
    return tables


target_summaries = summarize_runs_by_target(raw_results)
global_summary = summarize_runs(raw_results)

for target_name, summary_df in target_summaries.items():
    print(f"\n{'='*72}")
    print(f"  SUMMARY TABLE — Target Model: {target_name}")
    print(f"{'='*72}")
    display(summary_df)

if len(target_summaries) > 1:
    print(f"\n{'='*72}")
    print(f"  GLOBAL SUMMARY (all targets combined)")
    print(f"{'='*72}")
    display(global_summary)

raw_path = RESULTS_DIR / f"{CFG.dataset_name}_multi_target_per_seed_results.csv"
global_summary_path = RESULTS_DIR / f"{CFG.dataset_name}_multi_target_summary_auc_fpr95_cohend.csv"
vilu_ablation_path = RESULTS_DIR / f"{CFG.dataset_name}_multi_target_vilu_ablation_results.csv"
json_path = RESULTS_DIR / f"{CFG.dataset_name}_multi_target_results.json"

raw_results.to_csv(raw_path, index=False)
global_summary.to_csv(global_summary_path, index=False)
pd.DataFrame(vilu_ablation_rows).to_csv(vilu_ablation_path, index=False)

per_target_paths = []
for target_name, summary_df in target_summaries.items():
    p = RESULTS_DIR / f"{CFG.dataset_name}_{target_name}_summary_auc_fpr95_cohend.csv"
    summary_df.to_csv(p, index=False)
    per_target_paths.append(p)

with open(json_path, "w") as f:
    json.dump(
        {
            "config": asdict(CFG),
            "per_seed": raw_results.to_dict(orient="records"),
            "vilu_ablation": vilu_ablation_rows,
            "global_summary": global_summary.to_dict(orient="records"),
            "per_target_summaries": {
                target: df.to_dict(orient="records")
                for target, df in target_summaries.items()
            },
        },
        f, indent=2, default=str,
    )

plot_paths = []
for target_name, final_plot_registry in final_plot_registries.items():
    target_plot_path = RESULTS_DIR / f"{CFG.dataset_name}_{target_name}_uncertainty_distributions.png"
    fig = plot_uncertainty_distributions(
        final_plot_registry,
        title_suffix=f"{CFG.dataset_name} test set ({target_name} failures)",
    )
    fig.savefig(target_plot_path, dpi=200, bbox_inches="tight")
    plot_paths.append(target_plot_path)

print("\nSaved:")
for p in [raw_path, global_summary_path, vilu_ablation_path, json_path, *per_target_paths, *plot_paths]:
    print(f"  {p}")